# 01 — Подготовка признаков

## Temporal features

In [1]:
from __future__ import annotations

import os
os.environ.setdefault("TOKENIZERS_PARALLELISM", "false")

from dataclasses import dataclass
from datetime import date, datetime, timedelta
from pathlib import Path
import gc
import hashlib
import json
import math
import pickle
import random
import time
from typing import Iterable

import numpy as np
import polars as pl
import torch
from torch import nn
from torch.nn import functional as F
from torch.utils.data import ConcatDataset, DataLoader, Dataset

from scipy.optimize import linear_sum_assignment, minimize
from scipy.special import betaln, gammaln
from scipy.stats import rankdata
from IPython.display import display

try:
    from archetypes import AA
except Exception as e:
    raise RuntimeError(
        "Не установлен archetypes==0.12.2. Выполни: uv add archetypes==0.12.2"
    ) from e

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

CUDA_AVAILABLE = bool(torch.cuda.is_available() and torch.version.cuda is not None)
if not CUDA_AVAILABLE:
    raise RuntimeError("Нужна CUDA-сборка PyTorch и доступная NVIDIA GPU")

DEVICE = torch.device("cuda")
torch.cuda.manual_seed_all(SEED)
torch.set_float32_matmul_precision("high")

print("torch :", torch.__version__)
print("CUDA  :", torch.version.cuda)
print("GPU   :", torch.cuda.get_device_name(0))

assert hasattr(AA, "transform"), "Нужна версия archetypes с AA.transform"
print("Polars:", pl.__version__)


Using NumPy backend
torch : 2.12.1+cu126
CUDA  : 12.6
GPU   : NVIDIA GeForce RTX 4070 SUPER
Polars: 1.43.2


In [2]:
PROJECT_DIR = Path.cwd()

def _first_existing(*paths: Path) -> Path:
    for p in paths:
        if p.exists():
            return p
    return paths[0]

TRAIN_PATH = _first_existing(
    PROJECT_DIR / "data" / "train.parquet",
    PROJECT_DIR / "train.parquet",
)
SAMPLE_PATH = _first_existing(
    PROJECT_DIR / "data" / "sample_submit.csv",
    PROJECT_DIR / "data" / "sample_submission.csv",
    PROJECT_DIR / "sample_submit.csv",
    PROJECT_DIR / "sample_submission.csv",
)

FINAL_FEATURE_ROOT = PROJECT_DIR / "artifacts" / "prepare_features"
ARTIFACT_ROOT = FINAL_FEATURE_ROOT / "_cache"
MODEL_DIR = ARTIFACT_ROOT / "models"
TRAJECTORY_DIR = ARTIFACT_ROOT / "mean_daily_trajectories"
PRED_DIR = ARTIFACT_ROOT / "predictions"
ARCH_DIR = ARTIFACT_ROOT / "archetype_pit"
CACHE_DIR = ARTIFACT_ROOT / "cache"
REPORT_DIR = ARTIFACT_ROOT / "reports"

for p in [ARTIFACT_ROOT, MODEL_DIR, TRAJECTORY_DIR, PRED_DIR, ARCH_DIR, CACHE_DIR, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

HORIZON = 30
CONTEXT_DAYS = 180
MIN_OBS_CONTEXT_DAYS = 14

EXPECTED_MIN_DATE = date(2025, 1, 1)
FINAL_ANCHOR = date(2026, 2, 13)
EXPECTED_USERS = 250_000

CANONICAL_ANCHORS = [
    date(2025, 2, 13), date(2025, 3, 26), date(2025, 5, 7),
    date(2025, 6, 18), date(2025, 7, 30), date(2025, 9, 10),
    date(2025, 10, 22), date(2025, 11, 12), date(2025, 12, 3),
    date(2025, 12, 15), date(2026, 1, 14),
]
PRIMARY_FOLDS = [
    date(2025, 7, 30), date(2025, 9, 10),
    date(2025, 10, 22), date(2026, 1, 14),
]
STRESS_FOLDS = [date(2025, 12, 3), date(2025, 12, 15)]

FULL_REFRESH_ANCHORS = [
    date(2025, 2, 13),
    date(2025, 5, 7),
    date(2025, 7, 30),
    date(2025, 10, 22),
    date(2026, 1, 14),
    FINAL_ANCHOR,
]

POC_PRED_ANCHORS = sorted(set(PRIMARY_FOLDS + STRESS_FOLDS + [FINAL_ANCHOR]))

RUN_MODE = "full_export"
ENABLED_FAMILIES = ["gru", "tcn"]

TRAIN_USERS = 20_000
MAX_TRAIN_ORIGINS = 6
TRAIN_ORIGIN_STRIDE_DAYS = 28

BATCH_SIZE = 512
INFER_CHUNK_USERS = 2_000
NUM_WORKERS = 0
EPOCHS = 7
LEARNING_RATE = 3e-4
WEIGHT_DECAY = 1e-4
GRAD_CLIP = 1.0

GRU_HIDDEN = 96
GRU_LAYERS = 2
TCN_HIDDEN = 96
TCN_LEVELS = 4
DROPOUT = 0.10

LOSS_W_ANY_ORDER = 0.90
LOSS_W_ANY_ACTIVE = 0.20
LOSS_W_ORDER_DAY = 0.70
LOSS_W_ACTIVE_DAY = 0.35
LOSS_W_SEARCH_DAY = 0.15
LOSS_W_CAT_DAY = 0.10
LOSS_W_CART_DAY = 0.25
LOSS_W_POS_ORDERS = 0.65
LOSS_W_POS_CARTS = 0.20
LOSS_W_POS_SEARCHES = 0.15
LOSS_W_AGG_ORDER_DAYS = 0.80
LOSS_W_AGG_ACTIVE_DAYS = 0.45
LOSS_W_AGG_CART_DAYS = 0.20
LOSS_W_AGG_SEARCH_DAYS = 0.10
LOSS_W_AGG_ORDERS = 1.00
LOSS_W_AGG_CARTS = 0.25
LOSS_W_AGG_SEARCHES = 0.15

ARCH_K5 = 5
ARCH_K6 = 6
POSTNY_REFINEMENT_START = date(2026, 1, 12)
NY_REGIME_START = date(2025, 12, 15)
NY_REGIME_END = date(2026, 1, 11)

CORE_ARCH_FEATURES = [
    "search_day_propensity",
    "cat_day_propensity",
    "searches_per_active_day",
    "cart_day_propensity",
    "carts_per_cart_day",
    "order_vs_cart_balance",
    "order_day_propensity",
    "orders_per_order_day",
]

CORRECTED_INTERVAL_SPECS = [
    (date(2025, 1, 12), date(2025, 2, 13), "ordinary_0"),
    (date(2025, 2, 14), date(2025, 3, 26), "ordinary_1"),
    (date(2025, 3, 27), date(2025, 5, 7), "ordinary_2"),
    (date(2025, 5, 8), date(2025, 6, 18), "ordinary_3"),
    (date(2025, 6, 19), date(2025, 7, 30), "ordinary_4"),
    (date(2025, 7, 31), date(2025, 9, 10), "ordinary_5"),
    (date(2025, 9, 11), date(2025, 10, 22), "ordinary_6"),
    (date(2025, 10, 23), date(2025, 11, 12), "ordinary_7"),
    (date(2025, 11, 13), date(2025, 12, 14), "pre_ny_32d"),
    (date(2026, 1, 12), date(2026, 2, 13), "post_ny_33d"),
]

PIT_AA_SAMPLE_PER_INTERVAL = 3_000
PIT_AA_MAX_ITER = 450
PIT_AA_N_INIT = 2
PIT_AA_TOL = 1e-4
PIT_AA_METHOD = "nnls"
PIT_AA_TRANSFORM_CHUNK = 100_000
EB_FIT_SAMPLE = 120_000

MIN_FIT_ACTIVE_DAYS = 5
MIN_WEIGHT_ACTIVE_DAYS = 3
FUTURE_ACTIVE_RANK_MIN_EXPECTED_DAYS = 0.5
FUTURE_WEIGHT_MIN_EXPECTED_ACTIVE_DAYS = 3.0

FORCE_RETRAIN_TEMPORAL = False
FORCE_REPREDICT_TRAJECTORIES = False
FORCE_REBUILD_ARCHETYPE_BASES = False
FORCE_REBUILD_PREDICTION_PARQUETS = False
SAVE_MEAN_DAILY_TRAJECTORIES = True

if RUN_MODE not in {"poc", "full_export"}:
    raise ValueError(RUN_MODE)

PRED_ANCHORS = (
    POC_PRED_ANCHORS
    if RUN_MODE == "poc"
    else sorted(set(CANONICAL_ANCHORS + [FINAL_ANCHOR]))
)

print("RUN_MODE          :", RUN_MODE)
print("families          :", ENABLED_FAMILIES)
print("prediction anchors:", PRED_ANCHORS)
print("artifact root     :", ARTIFACT_ROOT.resolve())


RUN_MODE          : full_export
families          : ['gru', 'tcn']
prediction anchors: [datetime.date(2025, 2, 13), datetime.date(2025, 3, 26), datetime.date(2025, 5, 7), datetime.date(2025, 6, 18), datetime.date(2025, 7, 30), datetime.date(2025, 9, 10), datetime.date(2025, 10, 22), datetime.date(2025, 11, 12), datetime.date(2025, 12, 3), datetime.date(2025, 12, 15), datetime.date(2026, 1, 14), datetime.date(2026, 2, 13)]
artifact root     : D:\baboteek_holmotech\artifacts\prepare_features\_cache


In [3]:
HOLIDAY_CODES = {
    221, 222, 223,
    306, 307, 308,
    429, 430, 501,
    507, 508, 509,
    610, 611, 612,
    1102, 1103, 1104,
}

def monthday_code(d: date) -> int:
    return d.month * 100 + d.day

def is_newyear_core_day(d: date) -> bool:
    return (d.month == 12 and d.day >= 15) or (d.month == 1 and d.day <= 3)

def is_newyear_regime_day(d: date) -> bool:
    return NY_REGIME_START <= d <= NY_REGIME_END

def is_special_day(d: date) -> bool:
    return monthday_code(d) in HOLIDAY_CODES or is_newyear_core_day(d)

def _date_calendar_row(d: date, horizon_pos: float = 0.0) -> list[float]:
    dow = d.weekday()
    dom = d.day
    doy = d.timetuple().tm_yday
    year_len = 366.0 if (d.year % 4 == 0 and (d.year % 100 != 0 or d.year % 400 == 0)) else 365.0
    return [
        math.sin(2 * math.pi * dow / 7.0),
        math.cos(2 * math.pi * dow / 7.0),
        math.sin(2 * math.pi * (dom - 1) / 31.0),
        math.cos(2 * math.pi * (dom - 1) / 31.0),
        math.sin(2 * math.pi * (doy - 1) / year_len),
        math.cos(2 * math.pi * (doy - 1) / year_len),
        float(dow >= 5),
        float(is_special_day(d)),
        float(is_newyear_core_day(d)),
        float(is_newyear_regime_day(d)),
        float(horizon_pos),
    ]

def calendar_matrix(start: date, length: int, *, future: bool) -> np.ndarray:
    rows = []
    for i in range(length):
        d = start + timedelta(days=i)
        pos = (i + 1) / max(length, 1) if future else 0.0
        rows.append(_date_calendar_row(d, pos))
    return np.asarray(rows, dtype=np.float32)

PAST_CAL_DIM = 10
FUTURE_CAL_DIM = 11

def past_calendar_for_origin(origin: date, context_days: int = CONTEXT_DAYS) -> np.ndarray:
    start = origin - timedelta(days=context_days - 1)
    return calendar_matrix(start, context_days, future=False)[:, :PAST_CAL_DIM]

def future_calendar_for_origin(origin: date, horizon: int = HORIZON) -> np.ndarray:
    return calendar_matrix(origin + timedelta(days=1), horizon, future=True)


In [4]:
if not TRAIN_PATH.exists():
    raise FileNotFoundError(f"Не найден {TRAIN_PATH}")
if not SAMPLE_PATH.exists():
    raise FileNotFoundError("Не найден sample_submit.csv / sample_submission.csv")

REQUIRED_TRAIN_COLS = [
    "event_date", "user_id", "search", "cat", "to_cart", "to_ord", "searches",
]

sample = pl.read_csv(SAMPLE_PATH)
if "user_id" not in sample.columns:
    raise RuntimeError("sample submission не содержит user_id")

all_users = sample.select(pl.col("user_id").cast(pl.Int64)).unique(maintain_order=True)
ALL_USER_IDS = all_users["user_id"].to_numpy()

if len(ALL_USER_IDS) != EXPECTED_USERS:
    raise RuntimeError(f"Ожидалось {EXPECTED_USERS} users, найдено {len(ALL_USER_IDS)}")
if len(np.unique(ALL_USER_IDS)) != len(ALL_USER_IDS):
    raise RuntimeError("duplicate user_id в sample submission")

schema = pl.read_parquet_schema(TRAIN_PATH)
missing = [c for c in REQUIRED_TRAIN_COLS if c not in schema]
if missing:
    raise RuntimeError(f"train.parquet missing columns: {missing}")

daily = (
    pl.scan_parquet(TRAIN_PATH)
    .select(REQUIRED_TRAIN_COLS)
    .with_columns([
        pl.col("event_date").cast(pl.Date),
        pl.col("user_id").cast(pl.Int64),
        pl.col("search").fill_null(0).cast(pl.Float32),
        pl.col("cat").fill_null(0).cast(pl.Float32),
        pl.col("to_cart").fill_null(0).cast(pl.Float32),
        pl.col("to_ord").fill_null(0).cast(pl.Float32),
        pl.col("searches").fill_null(0).cast(pl.Float32),
    ])
    .collect(engine="streaming")
    .sort(["event_date", "user_id"])
)

bounds = daily.select([
    pl.col("event_date").min().alias("min_date"),
    pl.col("event_date").max().alias("max_date"),
    pl.len().alias("rows"),
])

MIN_DATE = bounds["min_date"][0]
MAX_DATE = bounds["max_date"][0]

print(bounds)
print("users in sample:", len(ALL_USER_IDS))

if MAX_DATE != FINAL_ANCHOR:
    raise RuntimeError(f"Unexpected max date {MAX_DATE}; expected {FINAL_ANCHOR}")
if MIN_DATE > EXPECTED_MIN_DATE:
    raise RuntimeError(f"Unexpected min date {MIN_DATE}; expected <= {EXPECTED_MIN_DATE}")

rng = np.random.default_rng(SEED)
TRAIN_USER_IDS = np.sort(
    rng.choice(ALL_USER_IDS, size=min(TRAIN_USERS, len(ALL_USER_IDS)), replace=False)
)

USER_ORDER_HASH = hashlib.md5(
    np.asarray(ALL_USER_IDS, dtype=np.int64).tobytes()
).hexdigest()[:12]

np.save(ARTIFACT_ROOT / "canonical_user_ids.npy", ALL_USER_IDS)

print("train user sample:", TRAIN_USER_IDS.shape)
print("user order hash  :", USER_ORDER_HASH)


shape: (1, 3)
┌────────────┬────────────┬──────────┐
│ min_date   ┆ max_date   ┆ rows     │
│ ---        ┆ ---        ┆ ---      │
│ date       ┆ date       ┆ u32      │
╞════════════╪════════════╪══════════╡
│ 2025-01-01 ┆ 2026-02-13 ┆ 30631006 │
└────────────┴────────────┴──────────┘
users in sample: 250000
train user sample: (20000,)
user order hash  : d900373f804d


In [5]:
def latest_refresh_for_anchor(anchor: date) -> date:
    valid = [r for r in FULL_REFRESH_ANCHORS if r <= anchor]
    if not valid:
        raise RuntimeError(f"No refresh <= anchor {anchor}")
    return max(valid)

def training_origins_for_cutoff(cutoff: date) -> list[date]:
    latest_origin = cutoff - timedelta(days=HORIZON)
    earliest_origin = EXPECTED_MIN_DATE + timedelta(days=MIN_OBS_CONTEXT_DAYS - 1)
    out = []
    a = latest_origin
    while a >= earliest_origin and len(out) < MAX_TRAIN_ORIGINS:
        out.append(a)
        a -= timedelta(days=TRAIN_ORIGIN_STRIDE_DAYS)
    out = sorted(out)
    if not out:
        raise RuntimeError(f"No completed training origins for cutoff {cutoff}")
    for origin in out:
        assert origin + timedelta(days=HORIZON) <= cutoff
    return out

def strict_oot_assert(cutoff: date, pred_anchor: date, train_origins: Iterable[date]) -> None:
    train_origins = list(train_origins)
    if cutoff > pred_anchor:
        raise AssertionError(f"fit cutoff {cutoff} > prediction anchor {pred_anchor}")
    max_target_end = max(o + timedelta(days=HORIZON) for o in train_origins)
    if max_target_end > cutoff:
        raise AssertionError(f"train target end {max_target_end} > fit cutoff {cutoff}")
    snapshot_target_start = pred_anchor + timedelta(days=1)
    if max_target_end >= snapshot_target_start and cutoff == pred_anchor:
        raise AssertionError("training target overlaps snapshot future target")

refresh_rows = []
for a in PRED_ANCHORS:
    r = latest_refresh_for_anchor(a)
    origins = training_origins_for_cutoff(r)
    strict_oot_assert(r, a, origins)
    completed_arch_intervals = [
        label for start, end, label in CORRECTED_INTERVAL_SPECS if end <= r
    ]
    refresh_rows.append({
        "prediction_anchor": str(a),
        "fit_cutoff": str(r),
        "n_train_origins": len(origins),
        "latest_train_origin": str(max(origins)),
        "latest_train_target_end": str(max(o + timedelta(days=HORIZON) for o in origins)),
        "n_completed_arch_intervals": len(completed_arch_intervals),
        "latest_arch_interval": completed_arch_intervals[-1] if completed_arch_intervals else None,
        "basis_has_completed_postny_33d": "post_ny_33d" in completed_arch_intervals,
    })

refresh_df = pl.DataFrame(refresh_rows)
display(refresh_df)
refresh_df.write_csv(REPORT_DIR / "strict_oot_refresh_plan.csv")


prediction_anchor,fit_cutoff,n_train_origins,latest_train_origin,latest_train_target_end,n_completed_arch_intervals,latest_arch_interval,basis_has_completed_postny_33d
str,str,i64,str,str,i64,str,bool
"""2025-02-13""","""2025-02-13""",1,"""2025-01-14""","""2025-02-13""",1,"""ordinary_0""",false
"""2025-03-26""","""2025-02-13""",1,"""2025-01-14""","""2025-02-13""",1,"""ordinary_0""",false
"""2025-05-07""","""2025-05-07""",3,"""2025-04-07""","""2025-05-07""",3,"""ordinary_2""",false
"""2025-06-18""","""2025-05-07""",3,"""2025-04-07""","""2025-05-07""",3,"""ordinary_2""",false
"""2025-07-30""","""2025-07-30""",6,"""2025-06-30""","""2025-07-30""",5,"""ordinary_4""",false
…,…,…,…,…,…,…,…
"""2025-11-12""","""2025-10-22""",6,"""2025-09-22""","""2025-10-22""",7,"""ordinary_6""",false
"""2025-12-03""","""2025-10-22""",6,"""2025-09-22""","""2025-10-22""",7,"""ordinary_6""",false
"""2025-12-15""","""2025-10-22""",6,"""2025-09-22""","""2025-10-22""",7,"""ordinary_6""",false


In [6]:
BEHAVIOUR_NAMES = [
    "active", "search", "cat", "order_day",
    "log1p_to_ord", "log1p_to_cart", "log1p_searches",
]
BEHAVIOUR_DIM = len(BEHAVIOUR_NAMES)
INPUT_DIM = BEHAVIOUR_DIM + PAST_CAL_DIM
TARGET_DIM = 8

def _user_index_frame(user_ids: np.ndarray) -> pl.DataFrame:
    ids = np.asarray(user_ids, dtype=np.int64)
    return pl.DataFrame({
        "user_id": ids,
        "_row": np.arange(len(ids), dtype=np.int32),
    })

def _window_rows(user_ids: np.ndarray, start: date, end: date) -> pl.DataFrame:
    ids_df = _user_index_frame(user_ids)
    return (
        daily.filter(pl.col("event_date").is_between(start, end, closed="both"))
        .join(ids_df, on="user_id", how="inner")
        .group_by(["_row", "event_date"])
        .agg([
            pl.col("search").max().alias("search"),
            pl.col("cat").max().alias("cat"),
            pl.col("to_cart").sum().alias("to_cart"),
            pl.col("to_ord").sum().alias("to_ord"),
            pl.col("searches").sum().alias("searches"),
        ])
        .sort(["_row", "event_date"])
    )

def build_context_tensor(user_ids: np.ndarray, origin: date, context_days: int = CONTEXT_DAYS) -> np.ndarray:
    ids = np.asarray(user_ids, dtype=np.int64)
    start = origin - timedelta(days=context_days - 1)
    part = _window_rows(ids, start, origin)
    X = np.zeros((len(ids), context_days, BEHAVIOUR_DIM), dtype=np.float32)

    if part.height:
        rows = part["_row"].to_numpy().astype(np.int64, copy=False)
        offsets = (
            part.select((pl.col("event_date") - pl.lit(start)).dt.total_days().alias("offset"))["offset"]
            .to_numpy()
            .astype(np.int64, copy=False)
        )
        to_ord = part["to_ord"].to_numpy().astype(np.float32, copy=False)
        to_cart = part["to_cart"].to_numpy().astype(np.float32, copy=False)
        searches = part["searches"].to_numpy().astype(np.float32, copy=False)

        X[rows, offsets, 0] = 1.0
        X[rows, offsets, 1] = np.clip(part["search"].to_numpy(), 0, 1)
        X[rows, offsets, 2] = np.clip(part["cat"].to_numpy(), 0, 1)
        X[rows, offsets, 3] = (to_ord > 0).astype(np.float32)
        X[rows, offsets, 4] = np.log1p(np.clip(to_ord, 0, None))
        X[rows, offsets, 5] = np.log1p(np.clip(to_cart, 0, None))
        X[rows, offsets, 6] = np.log1p(np.clip(searches, 0, None))

    cal = past_calendar_for_origin(origin, context_days)
    cal3 = np.broadcast_to(cal[None, :, :], (len(ids), context_days, PAST_CAL_DIM))
    return np.concatenate([X, cal3], axis=2).astype(np.float32, copy=False)

def build_future_target_tensor(user_ids: np.ndarray, origin: date, horizon: int = HORIZON) -> np.ndarray:
    ids = np.asarray(user_ids, dtype=np.int64)
    start = origin + timedelta(days=1)
    end = origin + timedelta(days=horizon)
    if end > MAX_DATE:
        raise ValueError(f"Future target unavailable for origin={origin}: end={end} > {MAX_DATE}")

    part = _window_rows(ids, start, end)
    Y = np.zeros((len(ids), horizon, TARGET_DIM), dtype=np.float32)

    if part.height:
        rows = part["_row"].to_numpy().astype(np.int64, copy=False)
        offsets = (
            part.select((pl.col("event_date") - pl.lit(start)).dt.total_days().alias("offset"))["offset"]
            .to_numpy()
            .astype(np.int64, copy=False)
        )
        search = np.clip(part["search"].to_numpy().astype(np.float32), 0, 1)
        cat = np.clip(part["cat"].to_numpy().astype(np.float32), 0, 1)
        to_ord = part["to_ord"].to_numpy().astype(np.float32, copy=False)
        to_cart = part["to_cart"].to_numpy().astype(np.float32, copy=False)
        searches = part["searches"].to_numpy().astype(np.float32, copy=False)

        Y[rows, offsets, 0] = (to_ord > 0).astype(np.float32)
        Y[rows, offsets, 1] = 1.0
        Y[rows, offsets, 2] = search
        Y[rows, offsets, 3] = cat
        Y[rows, offsets, 4] = (to_cart > 0).astype(np.float32)
        Y[rows, offsets, 5] = np.log1p(np.clip(to_ord, 0, None))
        Y[rows, offsets, 6] = np.log1p(np.clip(to_cart, 0, None))
        Y[rows, offsets, 7] = np.log1p(np.clip(searches, 0, None))

    return Y

def future_calendar_batch(origin: date, n_users: int) -> np.ndarray:
    cal = future_calendar_for_origin(origin, HORIZON)
    return np.broadcast_to(cal[None, :, :], (n_users, HORIZON, FUTURE_CAL_DIM)).astype(np.float32, copy=False)


In [7]:
_real_daily = globals().get("daily")
_real_min_date = globals().get("MIN_DATE")
_real_max_date = globals().get("MAX_DATE")

syn_users = np.arange(1, 33, dtype=np.int64)
syn_start = date(2025, 1, 1)
syn_end = date(2025, 5, 30)
rng_syn = np.random.default_rng(123)

rows = []
for uid in syn_users:
    for k in range((syn_end - syn_start).days + 1):
        d = syn_start + timedelta(days=k)
        if rng_syn.random() < 0.28:
            order = int(rng_syn.random() < 0.12)
            carts = int(rng_syn.poisson(0.8))
            searches = int(rng_syn.poisson(2.0))
            rows.append({
                "event_date": d,
                "user_id": int(uid),
                "search": float(searches > 0),
                "cat": float(rng_syn.random() < 0.5),
                "to_cart": float(carts),
                "to_ord": float(rng_syn.poisson(1.4) if order else 0),
                "searches": float(searches),
            })

daily = pl.DataFrame(rows).with_columns(pl.col("event_date").cast(pl.Date))
MIN_DATE = syn_start
MAX_DATE = syn_end

syn_origin = date(2025, 4, 20)
X_syn = build_context_tensor(syn_users, syn_origin, CONTEXT_DAYS)
Y_syn = build_future_target_tensor(syn_users, syn_origin, HORIZON)
FC_syn = future_calendar_batch(syn_origin, len(syn_users))

assert X_syn.shape == (len(syn_users), CONTEXT_DAYS, INPUT_DIM)
assert Y_syn.shape == (len(syn_users), HORIZON, TARGET_DIM)
assert FC_syn.shape == (len(syn_users), HORIZON, FUTURE_CAL_DIM)

assert np.isfinite(X_syn).all()
assert np.isfinite(Y_syn).all()
assert np.isfinite(FC_syn).all()

assert (Y_syn[:, :, 0] <= Y_syn[:, :, 1] + 1e-6).all()
assert (Y_syn[:, :, 2] <= Y_syn[:, :, 1] + 1e-6).all()
assert (Y_syn[:, :, 3] <= Y_syn[:, :, 1] + 1e-6).all()
assert (Y_syn[:, :, 4] <= Y_syn[:, :, 1] + 1e-6).all()

daily = _real_daily
MIN_DATE = _real_min_date
MAX_DATE = _real_max_date
del _real_daily, _real_min_date, _real_max_date

print("SYNTHETIC DATA BUILDERS: OK", X_syn.shape, Y_syn.shape, FC_syn.shape)


SYNTHETIC DATA BUILDERS: OK (32, 180, 17) (32, 30, 8) (32, 30, 11)


In [8]:
DAILY_OUTPUT_DIM = 8

class SharedFutureHead(nn.Module):
    def __init__(self, encoded_dim: int, future_cal_dim: int, hidden: int = 96, dropout: float = 0.10):
        super().__init__()
        self.daily = nn.Sequential(
            nn.Linear(encoded_dim + future_cal_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, DAILY_OUTPUT_DIM),
        )
        self.any_order = nn.Sequential(
            nn.Linear(encoded_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )
        self.any_active = nn.Sequential(
            nn.Linear(encoded_dim, hidden),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden, 1),
        )

    def forward(self, encoded: torch.Tensor, future_cal: torch.Tensor):
        h = encoded[:, None, :].expand(-1, future_cal.shape[1], -1)
        daily_raw = self.daily(torch.cat([h, future_cal], dim=-1))
        any_order_logit = self.any_order(encoded).squeeze(-1)
        any_active_logit = self.any_active(encoded).squeeze(-1)
        return daily_raw, any_order_logit, any_active_logit

class GRUBehaviourForecaster(nn.Module):
    def __init__(self, input_dim: int = INPUT_DIM):
        super().__init__()
        self.norm = nn.LayerNorm(input_dim)
        self.encoder = nn.GRU(
            input_size=input_dim,
            hidden_size=GRU_HIDDEN,
            num_layers=GRU_LAYERS,
            batch_first=True,
            dropout=DROPOUT if GRU_LAYERS > 1 else 0.0,
        )
        self.head = SharedFutureHead(
            GRU_HIDDEN, FUTURE_CAL_DIM, hidden=GRU_HIDDEN, dropout=DROPOUT
        )

    def forward(self, x: torch.Tensor, future_cal: torch.Tensor):
        x = self.norm(x)
        _, h = self.encoder(x)
        encoded = h[-1]
        return self.head(encoded, future_cal)

class _TCNBlock(nn.Module):
    def __init__(self, channels: int, dilation: int, dropout: float):
        super().__init__()
        self.dilation = dilation
        self.conv1 = nn.Conv1d(channels, channels, kernel_size=3, dilation=dilation)
        self.conv2 = nn.Conv1d(channels, channels, kernel_size=3, dilation=dilation)
        self.norm1 = nn.GroupNorm(1, channels)
        self.norm2 = nn.GroupNorm(1, channels)
        self.dropout = nn.Dropout(dropout)

    def _causal(self, x: torch.Tensor, conv: nn.Conv1d) -> torch.Tensor:
        pad = self.dilation * (conv.kernel_size[0] - 1)
        return conv(F.pad(x, (pad, 0)))

    def forward(self, x: torch.Tensor):
        residual = x
        x = self._causal(x, self.conv1)
        x = self.dropout(F.gelu(self.norm1(x)))
        x = self._causal(x, self.conv2)
        x = self.dropout(F.gelu(self.norm2(x)))
        return x + residual

class TCNBehaviourForecaster(nn.Module):
    def __init__(self, input_dim: int = INPUT_DIM):
        super().__init__()
        self.input_proj = nn.Conv1d(input_dim, TCN_HIDDEN, kernel_size=1)
        self.blocks = nn.ModuleList([
            _TCNBlock(TCN_HIDDEN, dilation=2**i, dropout=DROPOUT)
            for i in range(TCN_LEVELS)
        ])
        self.head = SharedFutureHead(
            TCN_HIDDEN, FUTURE_CAL_DIM, hidden=TCN_HIDDEN, dropout=DROPOUT
        )

    def forward(self, x: torch.Tensor, future_cal: torch.Tensor):
        x = x.transpose(1, 2)
        x = self.input_proj(x)
        for block in self.blocks:
            x = block(x)
        encoded = x[:, :, -1]
        return self.head(encoded, future_cal)

def make_model(family: str) -> nn.Module:
    if family == "gru":
        return GRUBehaviourForecaster()
    if family == "tcn":
        return TCNBehaviourForecaster()
    raise ValueError(family)

DAILY_PRED_KEYS = [
    "p_order", "p_active", "p_search", "p_cat", "p_cart",
    "e_orders", "e_carts", "e_searches",
]

def raw_to_daily(
    daily_raw: torch.Tensor,
    any_order_logit: torch.Tensor,
    any_active_logit: torch.Tensor,
) -> dict[str, torch.Tensor]:

    p_order = torch.sigmoid(daily_raw[..., 0])
    p_active_raw = torch.sigmoid(daily_raw[..., 1])
    p_search = torch.sigmoid(daily_raw[..., 2])
    p_cat = torch.sigmoid(daily_raw[..., 3])
    p_cart = torch.sigmoid(daily_raw[..., 4])

    p_active = torch.maximum(
        p_active_raw,
        torch.maximum(
            torch.maximum(p_order, p_search),
            torch.maximum(p_cat, p_cart),
        ),
    )

    pos_orders_log = F.softplus(daily_raw[..., 5])
    pos_carts_log = F.softplus(daily_raw[..., 6])
    pos_searches_log = F.softplus(daily_raw[..., 7])

    e_orders = p_order * torch.expm1(torch.clamp(pos_orders_log, max=6.0))
    e_carts = p_cart * torch.expm1(torch.clamp(pos_carts_log, max=7.0))
    e_searches = p_search * torch.expm1(torch.clamp(pos_searches_log, max=8.0))

    p_any_order = torch.sigmoid(any_order_logit)
    p_any_active = torch.maximum(torch.sigmoid(any_active_logit), p_any_order)

    return {
        "p_order": p_order,
        "p_active": p_active,
        "p_search": p_search,
        "p_cat": p_cat,
        "p_cart": p_cart,
        "e_orders": e_orders,
        "e_carts": e_carts,
        "e_searches": e_searches,
        "p_any_order": p_any_order,
        "p_any_active": p_any_active,
    }


In [9]:
def _sqrt_pos_weight(y: torch.Tensor, max_weight: float = 12.0) -> torch.Tensor:
    rate = torch.clamp(y.mean(), min=1e-4, max=1 - 1e-4)
    return torch.sqrt((1 - rate) / rate).detach().clamp(1.0, max_weight)

def multitask_loss(
    daily_raw: torch.Tensor,
    any_order_logit: torch.Tensor,
    any_active_logit: torch.Tensor,
    y: torch.Tensor,
) -> tuple[torch.Tensor, dict[str, float]]:

    y_order = y[..., 0]
    y_active = y[..., 1]
    y_search = y[..., 2]
    y_cat = y[..., 3]
    y_cart = y[..., 4]
    y_orders_log = y[..., 5]
    y_carts_log = y[..., 6]
    y_searches_log = y[..., 7]

    y_any_order = (y_order.sum(dim=1) > 0).float()
    y_any_active = (y_active.sum(dim=1) > 0).float()

    loss_any_order = F.binary_cross_entropy_with_logits(any_order_logit, y_any_order)
    loss_any_active = F.binary_cross_entropy_with_logits(any_active_logit, y_any_active)

    loss_order_day = F.binary_cross_entropy_with_logits(
        daily_raw[..., 0], y_order, pos_weight=_sqrt_pos_weight(y_order)
    )
    loss_active_day = F.binary_cross_entropy_with_logits(daily_raw[..., 1], y_active)
    loss_search_day = F.binary_cross_entropy_with_logits(
        daily_raw[..., 2], y_search, pos_weight=_sqrt_pos_weight(y_search, 8.0)
    )
    loss_cat_day = F.binary_cross_entropy_with_logits(
        daily_raw[..., 3], y_cat, pos_weight=_sqrt_pos_weight(y_cat, 8.0)
    )
    loss_cart_day = F.binary_cross_entropy_with_logits(
        daily_raw[..., 4], y_cart, pos_weight=_sqrt_pos_weight(y_cart, 10.0)
    )

    pred_orders_log = F.softplus(daily_raw[..., 5])
    pred_carts_log = F.softplus(daily_raw[..., 6])
    pred_searches_log = F.softplus(daily_raw[..., 7])

    order_mask = y_order > 0.5
    cart_mask = y_cart > 0.5
    search_mask = y_search > 0.5

    loss_pos_orders = (
        F.smooth_l1_loss(pred_orders_log[order_mask], y_orders_log[order_mask])
        if bool(order_mask.any()) else daily_raw[..., 5].sum() * 0.0
    )
    loss_pos_carts = (
        F.smooth_l1_loss(pred_carts_log[cart_mask], y_carts_log[cart_mask])
        if bool(cart_mask.any()) else daily_raw[..., 6].sum() * 0.0
    )
    loss_pos_searches = (
        F.smooth_l1_loss(pred_searches_log[search_mask], y_searches_log[search_mask])
        if bool(search_mask.any()) else daily_raw[..., 7].sum() * 0.0
    )

    decoded = raw_to_daily(daily_raw, any_order_logit, any_active_logit)

    true_order_days = y_order.sum(dim=1)
    true_active_days = y_active.sum(dim=1)
    true_search_days = y_search.sum(dim=1)
    true_cart_days = y_cart.sum(dim=1)
    true_orders = torch.expm1(y_orders_log).sum(dim=1)
    true_carts = torch.expm1(y_carts_log).sum(dim=1)
    true_searches = torch.expm1(y_searches_log).sum(dim=1)

    pred_order_days = decoded["p_order"].sum(dim=1)
    pred_active_days = decoded["p_active"].sum(dim=1)
    pred_search_days = decoded["p_search"].sum(dim=1)
    pred_cart_days = decoded["p_cart"].sum(dim=1)
    pred_orders = decoded["e_orders"].sum(dim=1)
    pred_carts = decoded["e_carts"].sum(dim=1)
    pred_searches = decoded["e_searches"].sum(dim=1)

    loss_agg_order_days = F.smooth_l1_loss(pred_order_days / HORIZON, true_order_days / HORIZON)
    loss_agg_active_days = F.smooth_l1_loss(pred_active_days / HORIZON, true_active_days / HORIZON)
    loss_agg_search_days = F.smooth_l1_loss(pred_search_days / HORIZON, true_search_days / HORIZON)
    loss_agg_cart_days = F.smooth_l1_loss(pred_cart_days / HORIZON, true_cart_days / HORIZON)

    loss_agg_orders = F.smooth_l1_loss(torch.log1p(pred_orders), torch.log1p(true_orders))
    loss_agg_carts = F.smooth_l1_loss(torch.log1p(pred_carts), torch.log1p(true_carts))
    loss_agg_searches = F.smooth_l1_loss(torch.log1p(pred_searches), torch.log1p(true_searches))

    total = (
        LOSS_W_ANY_ORDER * loss_any_order
        + LOSS_W_ANY_ACTIVE * loss_any_active
        + LOSS_W_ORDER_DAY * loss_order_day
        + LOSS_W_ACTIVE_DAY * loss_active_day
        + LOSS_W_SEARCH_DAY * loss_search_day
        + LOSS_W_CAT_DAY * loss_cat_day
        + LOSS_W_CART_DAY * loss_cart_day
        + LOSS_W_POS_ORDERS * loss_pos_orders
        + LOSS_W_POS_CARTS * loss_pos_carts
        + LOSS_W_POS_SEARCHES * loss_pos_searches
        + LOSS_W_AGG_ORDER_DAYS * loss_agg_order_days
        + LOSS_W_AGG_ACTIVE_DAYS * loss_agg_active_days
        + LOSS_W_AGG_CART_DAYS * loss_agg_cart_days
        + LOSS_W_AGG_SEARCH_DAYS * loss_agg_search_days
        + LOSS_W_AGG_ORDERS * loss_agg_orders
        + LOSS_W_AGG_CARTS * loss_agg_carts
        + LOSS_W_AGG_SEARCHES * loss_agg_searches
    )

    parts = {
        "total": float(total.detach().cpu()),
        "any_order": float(loss_any_order.detach().cpu()),
        "any_active": float(loss_any_active.detach().cpu()),
        "order_day": float(loss_order_day.detach().cpu()),
        "active_day": float(loss_active_day.detach().cpu()),
        "search_day": float(loss_search_day.detach().cpu()),
        "cat_day": float(loss_cat_day.detach().cpu()),
        "cart_day": float(loss_cart_day.detach().cpu()),
        "pos_orders": float(loss_pos_orders.detach().cpu()),
        "pos_carts": float(loss_pos_carts.detach().cpu()),
        "pos_searches": float(loss_pos_searches.detach().cpu()),
        "agg_order_days": float(loss_agg_order_days.detach().cpu()),
        "agg_active_days": float(loss_agg_active_days.detach().cpu()),
        "agg_search_days": float(loss_agg_search_days.detach().cpu()),
        "agg_cart_days": float(loss_agg_cart_days.detach().cpu()),
        "agg_orders": float(loss_agg_orders.detach().cpu()),
        "agg_carts": float(loss_agg_carts.detach().cpu()),
        "agg_searches": float(loss_agg_searches.detach().cpu()),
    }
    return total, parts


In [10]:
for family in ["gru", "tcn"]:
    model = make_model(family).to(DEVICE)

    xb = torch.from_numpy(X_syn[:8].copy()).to(
        device=DEVICE,
        dtype=torch.float32,
    )
    fb = torch.from_numpy(FC_syn[:8].copy()).to(
        device=DEVICE,
        dtype=torch.float32,
    )
    yb = torch.from_numpy(Y_syn[:8].copy()).to(
        device=DEVICE,
        dtype=torch.float32,
    )

    model.train()
    opt = torch.optim.AdamW(model.parameters(), lr=1e-4)
    opt.zero_grad(set_to_none=True)

    raw, any_order_logit, any_active_logit = model(xb, fb)

    assert raw.shape == (8, HORIZON, DAILY_OUTPUT_DIM)
    assert any_order_logit.shape == (8,)
    assert any_active_logit.shape == (8,)

    loss, parts = multitask_loss(
        raw,
        any_order_logit,
        any_active_logit,
        yb,
    )
    assert torch.isfinite(loss)

    loss.backward()
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    opt.step()

    decoded = raw_to_daily(
        raw,
        any_order_logit,
        any_active_logit,
    )

    for k, v in decoded.items():
        assert torch.isfinite(v).all(), k

    print(
        f"MODEL SMOKE {family}: OK | "
        f"loss={float(loss.detach().cpu()):.4f}"
    )

    del (
        model,
        opt,
        xb,
        fb,
        yb,
        raw,
        any_order_logit,
        any_active_logit,
        decoded,
    )

    if CUDA_AVAILABLE:
        torch.cuda.empty_cache()

gc.collect()

print("ALL SYNTHETIC MODEL TESTS: OK")

MODEL SMOKE gru: OK | loss=3.6491
MODEL SMOKE tcn: OK | loss=3.5068
ALL SYNTHETIC MODEL TESTS: OK


In [11]:
MODEL_VERSION = "aux_future_behaviour_archetypes_v2"

@dataclass
class OriginBlock:
    origin: date
    X: np.ndarray
    Y: np.ndarray
    future_cal: np.ndarray

class OriginDataset(Dataset):
    def __init__(self, block: OriginBlock):
        self.block = block

    def __len__(self):
        return self.block.X.shape[0]

    def __getitem__(self, idx: int):
        return (
            torch.from_numpy(self.block.X[idx]).float(),
            torch.from_numpy(self.block.future_cal).float(),
            torch.from_numpy(self.block.Y[idx]).float(),
        )

def _json_safe(obj):
    if isinstance(obj, dict):
        return {str(k): _json_safe(v) for k, v in obj.items()}
    if isinstance(obj, (list, tuple, set)):
        return [_json_safe(v) for v in obj]
    if isinstance(obj, (date, datetime, Path)):
        return str(obj)
    if isinstance(obj, np.generic):
        return obj.item()
    return obj

def stable_hash(obj) -> str:
    payload = json.dumps(
        _json_safe(obj),
        sort_keys=True,
        ensure_ascii=False,
        separators=(",", ":"),
    )
    return hashlib.md5(payload.encode("utf-8")).hexdigest()[:12]

def model_signature(family: str, cutoff: date, origins: list[date]) -> str:
    cfg = {
        "version": MODEL_VERSION,
        "family": family,
        "cutoff": cutoff,
        "origins": origins,
        "context_days": CONTEXT_DAYS,
        "horizon": HORIZON,
        "input_dim": INPUT_DIM,
        "target_dim": TARGET_DIM,
        "past_cal_dim": PAST_CAL_DIM,
        "future_cal_dim": FUTURE_CAL_DIM,
        "train_users": len(TRAIN_USER_IDS),
        "train_user_hash": hashlib.md5(
            np.asarray(TRAIN_USER_IDS, dtype=np.int64).tobytes()
        ).hexdigest()[:12],
        "epochs": EPOCHS,
        "lr": LEARNING_RATE,
        "wd": WEIGHT_DECAY,
        "gru_hidden": GRU_HIDDEN,
        "gru_layers": GRU_LAYERS,
        "tcn_hidden": TCN_HIDDEN,
        "tcn_levels": TCN_LEVELS,
        "dropout": DROPOUT,
        "loss_weights": [
            LOSS_W_ANY_ORDER, LOSS_W_ANY_ACTIVE,
            LOSS_W_ORDER_DAY, LOSS_W_ACTIVE_DAY, LOSS_W_SEARCH_DAY,
            LOSS_W_CAT_DAY, LOSS_W_CART_DAY,
            LOSS_W_POS_ORDERS, LOSS_W_POS_CARTS, LOSS_W_POS_SEARCHES,
            LOSS_W_AGG_ORDER_DAYS, LOSS_W_AGG_ACTIVE_DAYS,
            LOSS_W_AGG_CART_DAYS, LOSS_W_AGG_SEARCH_DAYS,
            LOSS_W_AGG_ORDERS, LOSS_W_AGG_CARTS, LOSS_W_AGG_SEARCHES,
        ],
    }
    return stable_hash(cfg)

def model_cache_path(family: str, cutoff: date, origins: list[date]) -> Path:
    sig = model_signature(family, cutoff, origins)
    return MODEL_DIR / f"{family}_cutoff_{cutoff}_{sig}.pt"

def build_training_blocks(cutoff: date) -> list[OriginBlock]:
    origins = training_origins_for_cutoff(cutoff)
    blocks = []

    for j, origin in enumerate(origins, 1):
        t0 = time.time()
        X = build_context_tensor(TRAIN_USER_IDS, origin, CONTEXT_DAYS).astype(np.float16)
        Y = build_future_target_tensor(TRAIN_USER_IDS, origin, HORIZON).astype(np.float16)
        fc = future_calendar_for_origin(origin, HORIZON).astype(np.float32)

        blocks.append(OriginBlock(origin=origin, X=X, Y=Y, future_cal=fc))
        print(
            f"  block {j}/{len(origins)} origin={origin} "
            f"X={X.shape} Y={Y.shape} sec={time.time()-t0:.1f}"
        )

    return blocks


In [12]:
def train_or_load_model(family: str, cutoff: date) -> tuple[nn.Module, dict]:
    origins = training_origins_for_cutoff(cutoff)
    strict_oot_assert(cutoff, cutoff, origins)
    path = model_cache_path(family, cutoff, origins)

    model = make_model(family).to(DEVICE)

    if path.exists() and not FORCE_RETRAIN_TEMPORAL:
        payload = torch.load(path, map_location=DEVICE, weights_only=False)
        expected_sig = model_signature(family, cutoff, origins)
        if payload.get("signature") != expected_sig:
            raise RuntimeError(f"cache signature mismatch: {path}")
        model.load_state_dict(payload["state_dict"])
        model.eval()
        print("loaded:", path.name)
        return model, payload["meta"]

    print(f"\nTRAIN {family} cutoff={cutoff} origins={origins}")

    blocks = build_training_blocks(cutoff)
    dataset = ConcatDataset([OriginDataset(b) for b in blocks])
    loader = DataLoader(
        dataset,
        batch_size=BATCH_SIZE,
        shuffle=True,
        num_workers=NUM_WORKERS,
        pin_memory=CUDA_AVAILABLE,
        drop_last=False,
    )

    opt = torch.optim.AdamW(
        model.parameters(),
        lr=LEARNING_RATE,
        weight_decay=WEIGHT_DECAY,
    )

    history = []
    model.train()

    for epoch in range(1, EPOCHS + 1):
        t0 = time.time()
        sums = {}
        n_batches = 0

        for xb, fb, yb in loader:
            xb = xb.to(DEVICE, non_blocking=True)
            fb = fb.to(DEVICE, non_blocking=True)
            yb = yb.to(DEVICE, non_blocking=True)

            opt.zero_grad(set_to_none=True)

            raw, any_order_logit, any_active_logit = model(xb, fb)
            loss, parts = multitask_loss(
                raw, any_order_logit, any_active_logit, yb
            )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

            for k, v in parts.items():
                sums[k] = sums.get(k, 0.0) + float(v)
            n_batches += 1

        row = {
            "epoch": epoch,
            **{k: v / max(n_batches, 1) for k, v in sums.items()},
        }
        row["seconds"] = time.time() - t0
        history.append(row)

        print(
            f"  epoch {epoch:02d}/{EPOCHS} "
            f"loss={row['total']:.4f} sec={row['seconds']:.1f}"
        )

    meta = {
        "version": MODEL_VERSION,
        "family": family,
        "cutoff": str(cutoff),
        "train_origins": [str(o) for o in origins],
        "max_train_target_end": str(max(o + timedelta(days=HORIZON) for o in origins)),
        "train_users": len(TRAIN_USER_IDS),
        "history": history,
    }

    payload = {
        "signature": model_signature(family, cutoff, origins),
        "state_dict": {k: v.detach().cpu() for k, v in model.state_dict().items()},
        "meta": meta,
    }

    torch.save(payload, path)
    print("saved:", path.name)

    del blocks, dataset, loader, opt, payload
    gc.collect()
    if CUDA_AVAILABLE:
        torch.cuda.empty_cache()

    model.eval()
    return model, meta


In [13]:
WEEK_SLICES = [(0, 7), (7, 14), (14, 21), (21, 30)]

CORE_STATE_NAMES = [
    "p_any_order", "p_any_active", "p_inactive_30d", "p_active_no_order_30d",
    "future_order_days_30d", "future_to_ord_30d", "future_active_days_30d",
    "orders_per_order_day",
    "future_search_days_30d", "future_cat_days_30d", "future_cart_days_30d",
    "future_searches_30d", "future_to_cart_30d",
    "searches_per_active_day", "carts_per_cart_day",
    "order_days_w1", "order_days_w2", "order_days_w3", "order_days_w4",
    "active_days_w1", "active_days_w2", "active_days_w3", "active_days_w4",
    "order_days_trend", "active_days_trend", "weekend_order_share",
]

def trajectory_to_features(daily_pred: dict[str, np.ndarray], origin: date) -> dict[str, np.ndarray]:
    p_order = np.asarray(daily_pred["p_order"], dtype=np.float64)
    p_active = np.asarray(daily_pred["p_active"], dtype=np.float64)
    p_search = np.asarray(daily_pred["p_search"], dtype=np.float64)
    p_cat = np.asarray(daily_pred["p_cat"], dtype=np.float64)
    p_cart = np.asarray(daily_pred["p_cart"], dtype=np.float64)
    e_orders = np.asarray(daily_pred["e_orders"], dtype=np.float64)
    e_carts = np.asarray(daily_pred["e_carts"], dtype=np.float64)
    e_searches = np.asarray(daily_pred["e_searches"], dtype=np.float64)
    p_any_order = np.asarray(daily_pred["p_any_order"], dtype=np.float64)
    p_any_active = np.asarray(daily_pred["p_any_active"], dtype=np.float64)

    order_days = p_order.sum(axis=1)
    active_days = p_active.sum(axis=1)
    search_days = p_search.sum(axis=1)
    cat_days = p_cat.sum(axis=1)
    cart_days = p_cart.sum(axis=1)

    orders = e_orders.sum(axis=1)
    carts = e_carts.sum(axis=1)
    searches = e_searches.sum(axis=1)

    out = {
        "p_any_order": np.clip(p_any_order, 0, 1),
        "p_any_active": np.clip(p_any_active, 0, 1),
        "p_inactive_30d": np.clip(1.0 - p_any_active, 0, 1),
        "p_active_no_order_30d": np.clip(p_any_active - p_any_order, 0, 1),
        "future_order_days_30d": np.clip(order_days, 0, HORIZON),
        "future_to_ord_30d": np.clip(orders, 0, None),
        "future_active_days_30d": np.clip(active_days, 0, HORIZON),
        "orders_per_order_day": orders / np.maximum(order_days, 1e-3),
        "future_search_days_30d": np.clip(search_days, 0, HORIZON),
        "future_cat_days_30d": np.clip(cat_days, 0, HORIZON),
        "future_cart_days_30d": np.clip(cart_days, 0, HORIZON),
        "future_searches_30d": np.clip(searches, 0, None),
        "future_to_cart_30d": np.clip(carts, 0, None),
        "searches_per_active_day": searches / np.maximum(active_days, 1e-3),
        "carts_per_cart_day": carts / np.maximum(cart_days, 1e-3),
    }

    order_weeks = []
    active_weeks = []
    for wi, (s, e) in enumerate(WEEK_SLICES, 1):
        ow = p_order[:, s:e].sum(axis=1)
        aw = p_active[:, s:e].sum(axis=1)
        order_weeks.append(ow)
        active_weeks.append(aw)
        out[f"order_days_w{wi}"] = ow
        out[f"active_days_w{wi}"] = aw

    out["order_days_trend"] = (
        order_weeks[-1] - order_weeks[0]
    ) / np.maximum(order_days, 1.0)

    out["active_days_trend"] = (
        active_weeks[-1] - active_weeks[0]
    ) / np.maximum(active_days, 1.0)

    fut_dates = [origin + timedelta(days=i) for i in range(1, HORIZON + 1)]
    weekend = np.asarray([d.weekday() >= 5 for d in fut_dates], dtype=bool)
    out["weekend_order_share"] = (
        p_order[:, weekend].sum(axis=1) / np.maximum(order_days, 1e-3)
    )

    for k in out:
        out[k] = np.nan_to_num(
            out[k], nan=0.0, posinf=0.0, neginf=0.0
        ).astype(np.float32)

    return out

def truth_to_daily(Y: np.ndarray) -> dict[str, np.ndarray]:
    y_order = Y[..., 0]
    y_active = Y[..., 1]
    y_search = Y[..., 2]
    y_cat = Y[..., 3]
    y_cart = Y[..., 4]

    return {
        "p_order": y_order,
        "p_active": y_active,
        "p_search": y_search,
        "p_cat": y_cat,
        "p_cart": y_cart,
        "e_orders": np.expm1(Y[..., 5]),
        "e_carts": np.expm1(Y[..., 6]),
        "e_searches": np.expm1(Y[..., 7]),
        "p_any_order": (y_order.sum(axis=1) > 0).astype(np.float32),
        "p_any_active": (y_active.sum(axis=1) > 0).astype(np.float32),
    }

def truth_to_features(Y: np.ndarray, origin: date) -> dict[str, np.ndarray]:
    return trajectory_to_features(truth_to_daily(Y), origin)


In [14]:
def _mae(y, p) -> float:
    y = np.asarray(y, dtype=np.float64)
    p = np.asarray(p, dtype=np.float64)
    return float(np.mean(np.abs(y - p)))

def _rmsle_count(y, p) -> float:
    y = np.clip(np.asarray(y, dtype=np.float64), 0, None)
    p = np.clip(np.asarray(p, dtype=np.float64), 0, None)
    return float(np.sqrt(np.mean((np.log1p(y) - np.log1p(p)) ** 2)))

def _logloss_binary(y, p, eps: float = 1e-6) -> float:
    y = np.asarray(y, dtype=np.float64)
    p = np.clip(np.asarray(p, dtype=np.float64), eps, 1 - eps)
    return float(-np.mean(y * np.log(p) + (1 - y) * np.log(1 - p)))

def _brier(y, p) -> float:
    y = np.asarray(y, dtype=np.float64)
    p = np.asarray(p, dtype=np.float64)
    return float(np.mean((y - p) ** 2))

def evaluate_state(pred: dict[str, np.ndarray], truth: dict[str, np.ndarray]) -> dict[str, float]:
    return {
        "p_any_order_logloss": _logloss_binary(truth["p_any_order"], pred["p_any_order"]),
        "p_any_order_brier": _brier(truth["p_any_order"], pred["p_any_order"]),
        "p_any_active_logloss": _logloss_binary(truth["p_any_active"], pred["p_any_active"]),
        "p_any_active_brier": _brier(truth["p_any_active"], pred["p_any_active"]),
        "order_days_mae": _mae(truth["future_order_days_30d"], pred["future_order_days_30d"]),
        "active_days_mae": _mae(truth["future_active_days_30d"], pred["future_active_days_30d"]),
        "search_days_mae": _mae(truth["future_search_days_30d"], pred["future_search_days_30d"]),
        "cat_days_mae": _mae(truth["future_cat_days_30d"], pred["future_cat_days_30d"]),
        "cart_days_mae": _mae(truth["future_cart_days_30d"], pred["future_cart_days_30d"]),
        "to_ord_rmsle": _rmsle_count(truth["future_to_ord_30d"], pred["future_to_ord_30d"]),
        "to_cart_rmsle": _rmsle_count(truth["future_to_cart_30d"], pred["future_to_cart_30d"]),
        "searches_rmsle": _rmsle_count(truth["future_searches_30d"], pred["future_searches_30d"]),
    }

def build_truth_state_all_users(anchor: date, user_ids: np.ndarray) -> dict[str, np.ndarray]:
    ids = np.asarray(user_ids, dtype=np.int64)
    accum = {k: [] for k in CORE_STATE_NAMES}

    for start_idx in range(0, len(ids), INFER_CHUNK_USERS):
        stop_idx = min(start_idx + INFER_CHUNK_USERS, len(ids))
        chunk_ids = ids[start_idx:stop_idx]
        Y = build_future_target_tensor(chunk_ids, anchor, HORIZON)
        feat = truth_to_features(Y, anchor)
        for k in CORE_STATE_NAMES:
            accum[k].append(feat[k])

    return {
        k: np.concatenate(v).astype(np.float32, copy=False)
        for k, v in accum.items()
    }


In [15]:
def fit_beta_binomial_prior(x: np.ndarray, n: np.ndarray, seed: int = SEED):
    x = np.asarray(x, dtype=np.float64)
    n = np.asarray(n, dtype=np.float64)
    valid = (
        np.isfinite(x) & np.isfinite(n) & (n > 0)
        & (x >= 0) & (x <= n + 1e-9)
    )
    if valid.sum() < 50:
        return 1.0, 1.0

    xv, nv = x[valid], n[valid]
    if len(xv) > EB_FIT_SAMPLE:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(xv), size=EB_FIT_SAMPLE, replace=False)
        xv, nv = xv[idx], nv[idx]

    mu = float(np.clip(xv.sum() / max(nv.sum(), 1.0), 1e-4, 1 - 1e-4))
    kappa0 = 20.0
    z0 = np.log([mu * kappa0, (1 - mu) * kappa0])

    def nll(z):
        a, b = np.exp(z)
        ll = (
            gammaln(nv + 1.0) - gammaln(xv + 1.0) - gammaln(nv - xv + 1.0)
            + betaln(xv + a, nv - xv + b) - betaln(a, b)
        )
        return -float(np.sum(ll))

    res = minimize(
        nll, z0, method="L-BFGS-B",
        bounds=[
            (math.log(1e-3), math.log(1e6)),
            (math.log(1e-3), math.log(1e6)),
        ],
    )
    if not res.success or not np.all(np.isfinite(res.x)):
        return float(mu * kappa0), float((1 - mu) * kappa0)

    a, b = np.exp(res.x)
    return float(a), float(b)

def fit_gamma_poisson_prior(count: np.ndarray, exposure: np.ndarray, seed: int = SEED):
    x = np.asarray(count, dtype=np.float64)
    e = np.asarray(exposure, dtype=np.float64)
    valid = np.isfinite(x) & np.isfinite(e) & (e > 0) & (x >= 0)
    if valid.sum() < 50:
        return 1.0, 1.0

    xv, ev = x[valid], e[valid]
    if len(xv) > EB_FIT_SAMPLE:
        rng = np.random.default_rng(seed)
        idx = rng.choice(len(xv), size=EB_FIT_SAMPLE, replace=False)
        xv, ev = xv[idx], ev[idx]

    mu = float(max(xv.sum() / max(ev.sum(), 1e-9), 1e-6))
    a0 = 2.0
    b0 = a0 / mu
    z0 = np.log([a0, b0])

    def nll(z):
        a, b = np.exp(z)
        ll = (
            gammaln(xv + a) - gammaln(a) - gammaln(xv + 1.0)
            + a * np.log(b) + xv * np.log(ev)
            - (xv + a) * np.log(b + ev)
        )
        return -float(np.sum(ll))

    res = minimize(
        nll, z0, method="L-BFGS-B",
        bounds=[
            (math.log(1e-3), math.log(1e6)),
            (math.log(1e-6), math.log(1e8)),
        ],
    )
    if not res.success or not np.all(np.isfinite(res.x)):
        return float(a0), float(b0)

    a, b = np.exp(res.x)
    return float(a), float(b)

def beta_posterior(x, n, a, b):
    x = np.asarray(x, dtype=np.float64)
    n = np.asarray(n, dtype=np.float64)
    valid = np.isfinite(x) & np.isfinite(n) & (n > 0) & (x >= 0)
    x_safe = np.clip(x, 0, np.maximum(n, 0))

    mean = np.full(len(x), a / (a + b), dtype=np.float64)
    conf = np.zeros(len(x), dtype=np.float64)
    mean[valid] = (x_safe[valid] + a) / (n[valid] + a + b)
    conf[valid] = n[valid] / (n[valid] + a + b)
    return mean, conf

def gamma_poisson_posterior(count, exposure, a, b):
    x = np.asarray(count, dtype=np.float64)
    e = np.asarray(exposure, dtype=np.float64)
    valid = np.isfinite(x) & np.isfinite(e) & (e > 0) & (x >= 0)

    mean = np.full(len(x), a / b, dtype=np.float64)
    conf = np.zeros(len(x), dtype=np.float64)
    mean[valid] = (x[valid] + a) / (e[valid] + b)
    conf[valid] = e[valid] / (e[valid] + b)
    return mean, conf

def rank01(values: np.ndarray) -> np.ndarray:
    x = np.asarray(values, dtype=np.float64)
    finite = np.isfinite(x)
    if not finite.any():
        return np.full(len(x), 0.5, dtype=np.float32)
    med = float(np.nanmedian(x[finite]))
    x = np.where(finite, x, med)
    if len(x) == 1:
        return np.array([0.5], dtype=np.float32)
    return ((rankdata(x, method="average") - 1.0) / (len(x) - 1.0)).astype(np.float32)

def aggregate_ordinary_interval(start: date, end: date) -> pl.DataFrame:
    part = daily.filter(pl.col("event_date").is_between(start, end, closed="both"))
    return (
        part.group_by("user_id")
        .agg([
            pl.col("event_date").n_unique().alias("active_days"),
            pl.col("event_date").filter(pl.col("search") > 0).n_unique().alias("search_days"),
            pl.col("event_date").filter(pl.col("cat") > 0).n_unique().alias("cat_days"),
            pl.col("event_date").filter(pl.col("to_cart") > 0).n_unique().alias("cart_days"),
            pl.col("event_date").filter(pl.col("to_ord") > 0).n_unique().alias("order_days"),
            pl.col("searches").sum().alias("searches_sum"),
            pl.col("to_cart").sum().alias("to_cart_sum"),
            pl.col("to_ord").sum().alias("to_ord_sum"),
        ])
        .sort("user_id")
    )

def interval_rank_cache_path(end: date) -> Path:
    return CACHE_DIR / f"arch_rank_interval_end_{end}.parquet"

def interval_prior_cache_path(end: date) -> Path:
    return CACHE_DIR / f"arch_prior_interval_end_{end}.json"

def build_interval_rank_and_priors(start: date, end: date, label: str) -> tuple[pl.DataFrame, dict]:
    rank_path = interval_rank_cache_path(end)
    prior_path = interval_prior_cache_path(end)

    if rank_path.exists() and prior_path.exists() and not FORCE_REBUILD_ARCHETYPE_BASES:
        with open(prior_path, "r", encoding="utf-8") as f:
            priors = json.load(f)
        return pl.read_parquet(rank_path), priors

    agg = aggregate_ordinary_interval(start, end)

    active = agg["active_days"].to_numpy().astype(np.float64)
    search_days = agg["search_days"].to_numpy().astype(np.float64)
    cat_days = agg["cat_days"].to_numpy().astype(np.float64)
    cart_days = agg["cart_days"].to_numpy().astype(np.float64)
    order_days = agg["order_days"].to_numpy().astype(np.float64)
    searches = agg["searches_sum"].to_numpy().astype(np.float64)
    carts = agg["to_cart_sum"].to_numpy().astype(np.float64)
    orders = agg["to_ord_sum"].to_numpy().astype(np.float64)

    priors = {
        "interval_label": label,
        "start_date": str(start),
        "end_date": str(end),
        "usable_days": int((end - start).days + 1),
        "beta": {},
        "gamma": {},
    }

    beta_specs = {
        "search_day_propensity": (search_days, active),
        "cat_day_propensity": (cat_days, active),
        "cart_day_propensity": (cart_days, active),
        "order_day_propensity": (order_days, active),
        "order_vs_cart_balance": (orders, orders + carts),
    }
    gamma_specs = {
        "searches_per_active_day": (searches, active),
        "carts_per_cart_day": (carts, cart_days),
        "orders_per_order_day": (orders, order_days),
    }

    coords = {}
    confs = []

    for j, (name, (x, n)) in enumerate(beta_specs.items()):
        a, b = fit_beta_binomial_prior(x, n, seed=SEED + j + int(end.strftime("%j")))
        mean, conf = beta_posterior(x, n, a, b)
        coords[name] = mean
        confs.append(conf)
        priors["beta"][name] = {"a": a, "b": b}

    for j, (name, (x, e)) in enumerate(gamma_specs.items()):
        a, b = fit_gamma_poisson_prior(x, e, seed=SEED + 100 + j + int(end.strftime("%j")))
        mean, conf = gamma_poisson_posterior(x, e, a, b)
        coords[name] = mean
        confs.append(conf)
        priors["gamma"][name] = {"a": a, "b": b}

    additions = []
    for name in CORE_ARCH_FEATURES:
        additions.append(pl.Series(f"arch_eb__{name}", coords[name].astype(np.float32)))
        additions.append(pl.Series(f"arch_x__{name}", rank01(coords[name])))

    conf_mat = np.column_stack(confs)
    eb_conf = np.nanmean(conf_mat, axis=1)
    active_evidence = np.minimum(1.0, active / 10.0)
    measurement_conf = np.clip(0.5 * eb_conf + 0.5 * active_evidence, 0, 1)

    ranked = (
        agg.with_columns(additions)
        .with_columns([
            pl.Series("archetype_measurement_confidence", measurement_conf.astype(np.float32)),
            (pl.col("active_days") >= MIN_FIT_ACTIVE_DAYS).alias("fit_eligible"),
            (pl.col("active_days") >= MIN_WEIGHT_ACTIVE_DAYS).alias("weight_eligible"),
            pl.lit(label).alias("interval_label"),
            pl.lit(start).cast(pl.Date).alias("interval_start"),
            pl.lit(end).cast(pl.Date).alias("interval_end"),
        ])
    )

    ranked.write_parquet(rank_path, compression="zstd")
    with open(prior_path, "w", encoding="utf-8") as f:
        json.dump(priors, f, ensure_ascii=False, indent=2)

    return ranked, priors

RANK_COLS = [f"arch_x__{c}" for c in CORE_ARCH_FEATURES]

def cosine_similarity_matrix(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    A = np.asarray(A, dtype=np.float64)
    B = np.asarray(B, dtype=np.float64)
    An = A / np.maximum(np.linalg.norm(A, axis=1, keepdims=True), 1e-12)
    Bn = B / np.maximum(np.linalg.norm(B, axis=1, keepdims=True), 1e-12)
    return An @ Bn.T

def align_profiles(reference: np.ndarray, candidate: np.ndarray):
    sim = cosine_similarity_matrix(reference, candidate)
    rows, cols = linear_sum_assignment(-sim)
    perm = np.empty(len(rows), dtype=int)
    perm[rows] = cols
    aligned = candidate[perm]
    matched = sim[np.arange(len(reference)), perm]
    return perm, aligned, matched

def fit_aa(X: np.ndarray, k: int, seed: int) -> AA:
    model = AA(
        n_archetypes=k,
        method=PIT_AA_METHOD,
        max_iter=PIT_AA_MAX_ITER,
        tol=PIT_AA_TOL,
        n_init=PIT_AA_N_INIT,
        random_state=seed,
        verbose=False,
    )
    model.fit(np.ascontiguousarray(X, dtype=np.float64))
    return model

def pit_basis_cache_path(cutoff: date, k: int) -> Path:
    return ARCH_DIR / f"aa_k{k}_cutoff_{cutoff}.pkl"

PIT_ARCH = {}

def build_all_pit_archetype_bases(cutoffs: list[date]) -> dict:
    previous_aligned = {ARCH_K5: None, ARCH_K6: None}
    result = {}

    for cutoff in sorted(cutoffs):
        completed = [
            (start, end, label)
            for start, end, label in CORRECTED_INTERVAL_SPECS
            if end <= cutoff
        ]
        if not completed:
            raise RuntimeError(f"No completed ordinary interval for cutoff {cutoff}")

        ranked_parts = []
        latest_priors = None

        for start, end, label in completed:
            ranked_i, priors_i = build_interval_rank_and_priors(start, end, label)
            fit_i = ranked_i.filter(pl.col("fit_eligible"))
            if fit_i.height == 0:
                continue
            take = min(PIT_AA_SAMPLE_PER_INTERVAL, fit_i.height)
            sample_i = fit_i.sample(
                n=take,
                seed=SEED + int(end.strftime("%Y%m%d")),
                shuffle=True,
            )
            ranked_parts.append(sample_i.select(RANK_COLS))
            latest_priors = priors_i

        if not ranked_parts:
            raise RuntimeError(f"No PIT AA fit rows for cutoff {cutoff}")

        fit_frame = pl.concat(ranked_parts, how="vertical_relaxed")
        X_fit = fit_frame.to_numpy().astype(np.float64, copy=False)

        result[cutoff] = {
            "completed_intervals": [label for _, _, label in completed],
            "latest_priors": latest_priors,
            "models": {},
        }

        for k in [ARCH_K5, ARCH_K6]:
            cache_path = pit_basis_cache_path(cutoff, k)

            if cache_path.exists() and not FORCE_REBUILD_ARCHETYPE_BASES:
                with open(cache_path, "rb") as f:
                    payload = pickle.load(f)
                model = payload["model"]
                perm = np.asarray(payload["perm_to_aligned"], dtype=int)
                aligned = np.asarray(payload["aligned_profiles"], dtype=np.float64)
                matched = np.asarray(payload["alignment_cosines"], dtype=np.float64)
            else:
                model = fit_aa(
                    X_fit,
                    k,
                    seed=SEED + 10_000 * k + int(cutoff.strftime("%j")),
                )
                raw_profiles = np.asarray(model.archetypes_, dtype=np.float64)

                if previous_aligned[k] is None:
                    perm = np.arange(k, dtype=int)
                    aligned = raw_profiles.copy()
                    matched = np.ones(k, dtype=np.float64)
                else:
                    perm, aligned, matched = align_profiles(previous_aligned[k], raw_profiles)

                payload = {
                    "version": "pit_arch_v1",
                    "cutoff": str(cutoff),
                    "k": k,
                    "completed_intervals": [label for _, _, label in completed],
                    "model": model,
                    "perm_to_aligned": perm,
                    "aligned_profiles": aligned,
                    "alignment_cosines": matched,
                }
                with open(cache_path, "wb") as f:
                    pickle.dump(payload, f, protocol=pickle.HIGHEST_PROTOCOL)

            previous_aligned[k] = aligned.copy()
            result[cutoff]["models"][k] = {
                "model": model,
                "perm": perm,
                "aligned_profiles": aligned,
                "alignment_cosines": matched,
            }

            profile_df = pl.DataFrame({
                "archetype": np.arange(k, dtype=np.int16),
                **{
                    f"rank_profile__{f}": aligned[:, j]
                    for j, f in enumerate(CORE_ARCH_FEATURES)
                },
            })
            profile_df.write_csv(ARCH_DIR / f"profiles_k{k}_cutoff_{cutoff}.csv")

            print(
                f"PIT AA cutoff={cutoff} K{k} intervals={len(completed)} "
                f"fit_rows={len(X_fit)} align_mean={float(np.mean(matched)):.4f}"
            )

    return result

REQUIRED_REFRESHES = sorted({latest_refresh_for_anchor(a) for a in PRED_ANCHORS})
PIT_ARCH = build_all_pit_archetype_bases(REQUIRED_REFRESHES)


PIT AA cutoff=2025-02-13 K5 intervals=1 fit_rows=3000 align_mean=1.0000
PIT AA cutoff=2025-02-13 K6 intervals=1 fit_rows=3000 align_mean=1.0000
PIT AA cutoff=2025-05-07 K5 intervals=3 fit_rows=9000 align_mean=0.9990
PIT AA cutoff=2025-05-07 K6 intervals=3 fit_rows=9000 align_mean=0.9932
PIT AA cutoff=2025-07-30 K5 intervals=5 fit_rows=15000 align_mean=0.9996
PIT AA cutoff=2025-07-30 K6 intervals=5 fit_rows=15000 align_mean=0.8966
PIT AA cutoff=2025-10-22 K5 intervals=7 fit_rows=21000 align_mean=0.9998
PIT AA cutoff=2025-10-22 K6 intervals=7 fit_rows=21000 align_mean=0.9996
PIT AA cutoff=2026-01-14 K5 intervals=9 fit_rows=27000 align_mean=0.9999
PIT AA cutoff=2026-01-14 K6 intervals=9 fit_rows=27000 align_mean=0.8923
PIT AA cutoff=2026-02-13 K5 intervals=10 fit_rows=30000 align_mean=1.0000
PIT AA cutoff=2026-02-13 K6 intervals=10 fit_rows=30000 align_mean=1.0000


In [16]:
def masked_percentile_values(values: np.ndarray, reference_mask: np.ndarray) -> np.ndarray:
    x = np.asarray(values, dtype=np.float64)
    mask = np.asarray(reference_mask, dtype=bool) & np.isfinite(x)
    ref = x[mask]

    if len(ref) < 2:
        return np.full(len(x), 0.5, dtype=np.float32)

    ref = np.sort(ref)
    left = np.searchsorted(ref, x, side="left")
    right = np.searchsorted(ref, x, side="right")
    mid = 0.5 * (left + right - 1)
    out = mid / max(len(ref) - 1, 1)
    return np.clip(out, 0, 1).astype(np.float32)

def sufficient_from_daily(daily_pred: dict[str, np.ndarray]) -> dict[str, np.ndarray]:
    return {
        "active_days": np.asarray(daily_pred["p_active"], dtype=np.float64).sum(axis=1),
        "search_days": np.asarray(daily_pred["p_search"], dtype=np.float64).sum(axis=1),
        "cat_days": np.asarray(daily_pred["p_cat"], dtype=np.float64).sum(axis=1),
        "cart_days": np.asarray(daily_pred["p_cart"], dtype=np.float64).sum(axis=1),
        "order_days": np.asarray(daily_pred["p_order"], dtype=np.float64).sum(axis=1),
        "searches_sum": np.asarray(daily_pred["e_searches"], dtype=np.float64).sum(axis=1),
        "to_cart_sum": np.asarray(daily_pred["e_carts"], dtype=np.float64).sum(axis=1),
        "to_ord_sum": np.asarray(daily_pred["e_orders"], dtype=np.float64).sum(axis=1),
    }

def coords_from_sufficient(stats: dict[str, np.ndarray], priors: dict):
    active = np.asarray(stats["active_days"], dtype=np.float64)
    search_days = np.asarray(stats["search_days"], dtype=np.float64)
    cat_days = np.asarray(stats["cat_days"], dtype=np.float64)
    cart_days = np.asarray(stats["cart_days"], dtype=np.float64)
    order_days = np.asarray(stats["order_days"], dtype=np.float64)
    searches = np.asarray(stats["searches_sum"], dtype=np.float64)
    carts = np.asarray(stats["to_cart_sum"], dtype=np.float64)
    orders = np.asarray(stats["to_ord_sum"], dtype=np.float64)

    beta_inputs = {
        "search_day_propensity": (search_days, active),
        "cat_day_propensity": (cat_days, active),
        "cart_day_propensity": (cart_days, active),
        "order_day_propensity": (order_days, active),
        "order_vs_cart_balance": (orders, orders + carts),
    }
    gamma_inputs = {
        "searches_per_active_day": (searches, active),
        "carts_per_cart_day": (carts, cart_days),
        "orders_per_order_day": (orders, order_days),
    }

    coords = {}
    confs = []

    for name, (x, n) in beta_inputs.items():
        p = priors["beta"][name]
        mean, conf = beta_posterior(x, n, float(p["a"]), float(p["b"]))
        coords[name] = mean
        confs.append(conf)

    for name, (x, e) in gamma_inputs.items():
        p = priors["gamma"][name]
        mean, conf = gamma_poisson_posterior(x, e, float(p["a"]), float(p["b"]))
        coords[name] = mean
        confs.append(conf)

    conf_mat = np.column_stack(confs)
    eb_conf = np.nanmean(conf_mat, axis=1)
    active_evidence = np.minimum(1.0, active / 10.0)
    measurement_conf = np.clip(0.5 * eb_conf + 0.5 * active_evidence, 0, 1)

    return coords, measurement_conf

def rank_core_population(coords: dict[str, np.ndarray], active_reference_mask: np.ndarray) -> np.ndarray:
    cols = [
        masked_percentile_values(coords[name], active_reference_mask)
        for name in CORE_ARCH_FEATURES
    ]
    return np.column_stack(cols).astype(np.float64, copy=False)

def transform_aa_aligned(model: AA, perm: np.ndarray, X: np.ndarray) -> np.ndarray:
    parts = []
    for s in range(0, len(X), PIT_AA_TRANSFORM_CHUNK):
        w = np.asarray(
            model.transform(X[s:s + PIT_AA_TRANSFORM_CHUNK]),
            dtype=np.float64,
        )
        parts.append(w[:, perm])

    W = np.vstack(parts)
    W = np.clip(W, 0, None)
    W = W / np.maximum(W.sum(axis=1, keepdims=True), 1e-12)
    return W.astype(np.float32)

def normalized_entropy(W: np.ndarray) -> np.ndarray:
    x = np.clip(np.asarray(W, dtype=np.float64), 1e-12, 1)
    return (-np.sum(x * np.log(x), axis=1) / np.log(x.shape[1])).astype(np.float32)

def archetype_projection_from_sufficient(
    stats: dict[str, np.ndarray],
    cutoff: date,
    *,
    predicted: bool,
) -> dict[str, np.ndarray]:

    arch = PIT_ARCH[cutoff]
    priors = arch["latest_priors"]
    coords, measurement_conf = coords_from_sufficient(stats, priors)

    active_days = np.asarray(stats["active_days"], dtype=np.float64)

    if predicted:
        rank_ref_mask = active_days >= FUTURE_ACTIVE_RANK_MIN_EXPECTED_DAYS
        weight_eligible = active_days >= FUTURE_WEIGHT_MIN_EXPECTED_ACTIVE_DAYS
    else:
        rank_ref_mask = active_days > 0
        weight_eligible = active_days >= MIN_WEIGHT_ACTIVE_DAYS

    X_rank = rank_core_population(coords, rank_ref_mask)

    out = {
        "rank_reference_mask": rank_ref_mask,
        "weight_eligible": weight_eligible,
        "measurement_confidence": measurement_conf.astype(np.float32),
        "X_rank": X_rank.astype(np.float32),
    }

    for name in CORE_ARCH_FEATURES:
        out[f"eb__{name}"] = np.asarray(coords[name], dtype=np.float32)

    for j, name in enumerate(CORE_ARCH_FEATURES):
        out[f"rank__{name}"] = X_rank[:, j].astype(np.float32)

    for k in [ARCH_K5, ARCH_K6]:
        model_info = arch["models"][k]
        W = transform_aa_aligned(model_info["model"], model_info["perm"], X_rank)
        ent = normalized_entropy(W)

        out[f"k{k}_weights"] = W
        out[f"k{k}_dominant"] = np.argmax(W, axis=1).astype(np.int16)
        out[f"k{k}_dominant_weight"] = np.max(W, axis=1).astype(np.float32)
        out[f"k{k}_entropy"] = ent
        out[f"k{k}_effective_count"] = np.exp(ent * np.log(k)).astype(np.float32)

    return out

def build_truth_future_sufficient_all_users(anchor: date) -> dict[str, np.ndarray]:
    start = anchor + timedelta(days=1)
    end = anchor + timedelta(days=HORIZON)

    if end > MAX_DATE:
        raise ValueError(f"truth unavailable for {anchor}")

    part = daily.filter(pl.col("event_date").is_between(start, end, closed="both"))

    agg = (
        part.group_by("user_id")
        .agg([
            pl.col("event_date").n_unique().alias("active_days"),
            pl.col("event_date").filter(pl.col("search") > 0).n_unique().alias("search_days"),
            pl.col("event_date").filter(pl.col("cat") > 0).n_unique().alias("cat_days"),
            pl.col("event_date").filter(pl.col("to_cart") > 0).n_unique().alias("cart_days"),
            pl.col("event_date").filter(pl.col("to_ord") > 0).n_unique().alias("order_days"),
            pl.col("searches").sum().alias("searches_sum"),
            pl.col("to_cart").sum().alias("to_cart_sum"),
            pl.col("to_ord").sum().alias("to_ord_sum"),
        ])
    )

    zero_cols = [
        "active_days", "search_days", "cat_days", "cart_days", "order_days",
        "searches_sum", "to_cart_sum", "to_ord_sum",
    ]

    base = (
        all_users.join(agg, on="user_id", how="left", maintain_order="left")
        .with_columns([pl.col(c).fill_null(0) for c in zero_cols])
    )

    return {
        c: base[c].to_numpy().astype(np.float64, copy=False)
        for c in zero_cols
    }

def archetype_projection_to_frame(
    proj: dict[str, np.ndarray],
    anchor: date,
    cutoff: date,
) -> pl.DataFrame:

    cols = {
        "user_id": ALL_USER_IDS,
        "anchor_date": np.full(len(ALL_USER_IDS), np.datetime64(anchor, "D")),
        "arch_fit_cutoff": np.full(len(ALL_USER_IDS), np.datetime64(cutoff, "D")),
        "future_arch_measurement_confidence": proj["measurement_confidence"],
        "future_arch_weight_eligible": proj["weight_eligible"].astype(np.int8),
        "postny_k6_refinement_enabled": np.full(
            len(ALL_USER_IDS),
            int(anchor >= POSTNY_REFINEMENT_START),
            dtype=np.int8,
        ),
        "k6_basis_has_completed_postny_33d": np.full(
            len(ALL_USER_IDS),
            int("post_ny_33d" in PIT_ARCH[cutoff]["completed_intervals"]),
            dtype=np.int8,
        ),
    }

    for name in CORE_ARCH_FEATURES:
        cols[f"future_arch_eb__{name}"] = proj[f"eb__{name}"]
        cols[f"future_arch_rank__{name}"] = proj[f"rank__{name}"]

    for k in [ARCH_K5, ARCH_K6]:
        W = proj[f"k{k}_weights"]
        for j in range(k):
            cols[f"future_arch{k}_w_{j}"] = W[:, j]

        cols[f"future_arch{k}_dominant"] = proj[f"k{k}_dominant"]
        cols[f"future_arch{k}_dominant_weight"] = proj[f"k{k}_dominant_weight"]
        cols[f"future_arch{k}_entropy"] = proj[f"k{k}_entropy"]
        cols[f"future_arch{k}_effective_count"] = proj[f"k{k}_effective_count"]

    return pl.DataFrame(cols)


In [17]:
def _row_cosine(A: np.ndarray, B: np.ndarray) -> np.ndarray:
    A = np.asarray(A, dtype=np.float64)
    B = np.asarray(B, dtype=np.float64)
    num = np.sum(A * B, axis=1)
    den = np.linalg.norm(A, axis=1) * np.linalg.norm(B, axis=1)
    return np.divide(num, den, out=np.zeros_like(num), where=den > 1e-12)

def evaluate_archetype_projection(
    pred: dict[str, np.ndarray],
    truth: dict[str, np.ndarray],
    k: int,
) -> dict[str, float]:

    mask = np.asarray(truth["weight_eligible"], dtype=bool)

    if mask.sum() == 0:
        return {
            "rows": 0,
            "truth_weight_eligible_share": 0.0,
            "pred_weight_eligible_share": float(np.mean(pred["weight_eligible"])),
            "mean_weight_cosine": float("nan"),
            "median_weight_cosine": float("nan"),
            "weight_mae": float("nan"),
            "dominant_agreement": float("nan"),
            "rank_mae": float("nan"),
        }

    Wp = pred[f"k{k}_weights"][mask]
    Wt = truth[f"k{k}_weights"][mask]
    cos = _row_cosine(Wp, Wt)

    dom_p = np.argmax(Wp, axis=1)
    dom_t = np.argmax(Wt, axis=1)

    return {
        "rows": int(mask.sum()),
        "truth_weight_eligible_share": float(mask.mean()),
        "pred_weight_eligible_share": float(np.mean(pred["weight_eligible"])),
        "mean_weight_cosine": float(np.mean(cos)),
        "median_weight_cosine": float(np.median(cos)),
        "weight_mae": float(np.mean(np.abs(Wp - Wt))),
        "dominant_agreement": float(np.mean(dom_p == dom_t)),
        "rank_mae": float(np.mean(np.abs(pred["X_rank"][mask] - truth["X_rank"][mask]))),
    }


In [18]:
def trajectory_cache_path(anchor: date) -> Path:
    return TRAJECTORY_DIR / f"mean_daily_anchor_{anchor}.npz"

def ensemble_signature(cutoff: date) -> str:
    origins = training_origins_for_cutoff(cutoff)
    return stable_hash({
        "version": MODEL_VERSION,
        "cutoff": cutoff,
        "families": ENABLED_FAMILIES,
        "model_signatures": {
            f: model_signature(f, cutoff, origins)
            for f in ENABLED_FAMILIES
        },
        "user_order_hash": USER_ORDER_HASH,
        "horizon": HORIZON,
    })

def save_trajectory_cache(
    anchor: date,
    cutoff: date,
    mean_daily: dict[str, np.ndarray],
    mean_compact: dict[str, np.ndarray],
) -> None:
    if not SAVE_MEAN_DAILY_TRAJECTORIES:
        return

    payload = {
        "signature": np.asarray(ensemble_signature(cutoff)),
        "anchor": np.asarray(str(anchor)),
        "cutoff": np.asarray(str(cutoff)),
        "user_order_hash": np.asarray(USER_ORDER_HASH),
    }

    for k in DAILY_PRED_KEYS:
        payload["daily__" + k] = mean_daily[k].astype(np.float16)

    payload["head__p_any_order"] = mean_daily["p_any_order"].astype(np.float16)
    payload["head__p_any_active"] = mean_daily["p_any_active"].astype(np.float16)

    for k in CORE_STATE_NAMES:
        payload["compact__" + k] = mean_compact[k].astype(np.float32)

    np.savez_compressed(trajectory_cache_path(anchor), **payload)

def load_trajectory_cache(
    anchor: date,
    cutoff: date,
) -> tuple[dict[str, np.ndarray], dict[str, np.ndarray]] | None:

    p = trajectory_cache_path(anchor)

    if not p.exists() or FORCE_REPREDICT_TRAJECTORIES:
        return None

    z = np.load(p, allow_pickle=False)

    if str(z["signature"].item()) != ensemble_signature(cutoff):
        print("trajectory cache signature mismatch:", p.name)
        return None

    if str(z["user_order_hash"].item()) != USER_ORDER_HASH:
        print("trajectory cache user hash mismatch:", p.name)
        return None

    mean_daily = {
        k: z["daily__" + k].astype(np.float32)
        for k in DAILY_PRED_KEYS
    }
    mean_daily["p_any_order"] = z["head__p_any_order"].astype(np.float32)
    mean_daily["p_any_active"] = z["head__p_any_active"].astype(np.float32)

    mean_compact = {
        k: z["compact__" + k].astype(np.float32)
        for k in CORE_STATE_NAMES
    }

    return mean_daily, mean_compact

@torch.no_grad()
def predict_mean_ensemble_anchor(
    model_by_family: dict[str, nn.Module],
    anchor: date,
    cutoff: date,
) -> tuple[dict[str, np.ndarray], dict[str, np.ndarray]]:

    cached = load_trajectory_cache(anchor, cutoff)
    if cached is not None:
        print("loaded mean trajectory cache:", trajectory_cache_path(anchor).name)
        return cached

    n = len(ALL_USER_IDS)

    mean_daily = {
        k: np.zeros((n, HORIZON), dtype=np.float32)
        for k in DAILY_PRED_KEYS
    }
    mean_daily["p_any_order"] = np.zeros(n, dtype=np.float32)
    mean_daily["p_any_active"] = np.zeros(n, dtype=np.float32)

    compact_sum = {
        k: np.zeros(n, dtype=np.float64)
        for k in CORE_STATE_NAMES
    }

    n_families = len(ENABLED_FAMILIES)

    for family in ENABLED_FAMILIES:
        model = model_by_family[family]
        model.eval()

        print(f"  inference family={family}")
        t0 = time.time()

        for start_idx in range(0, n, INFER_CHUNK_USERS):
            stop_idx = min(start_idx + INFER_CHUNK_USERS, n)
            ids = ALL_USER_IDS[start_idx:stop_idx]

            X = build_context_tensor(ids, anchor, CONTEXT_DAYS)
            fc = future_calendar_batch(anchor, len(ids))

            xb = torch.from_numpy(X).float().to(DEVICE, non_blocking=True)
            fb = torch.from_numpy(fc).float().to(DEVICE, non_blocking=True)

            raw, any_order_logit, any_active_logit = model(xb, fb)
            decoded_t = raw_to_daily(raw, any_order_logit, any_active_logit)

            chunk_daily = {
                k: decoded_t[k].detach().cpu().numpy().astype(np.float32, copy=False)
                for k in DAILY_PRED_KEYS
            }
            chunk_daily["p_any_order"] = (
                decoded_t["p_any_order"].detach().cpu().numpy().astype(np.float32, copy=False)
            )
            chunk_daily["p_any_active"] = (
                decoded_t["p_any_active"].detach().cpu().numpy().astype(np.float32, copy=False)
            )

            for k in DAILY_PRED_KEYS:
                mean_daily[k][start_idx:stop_idx] += chunk_daily[k] / n_families
            mean_daily["p_any_order"][start_idx:stop_idx] += (
                chunk_daily["p_any_order"] / n_families
            )
            mean_daily["p_any_active"][start_idx:stop_idx] += (
                chunk_daily["p_any_active"] / n_families
            )

            family_compact = trajectory_to_features(chunk_daily, anchor)
            for k in CORE_STATE_NAMES:
                compact_sum[k][start_idx:stop_idx] += family_compact[k] / n_families

            del (
                X, fc, xb, fb, raw, any_order_logit, any_active_logit,
                decoded_t, chunk_daily, family_compact
            )
            if CUDA_AVAILABLE:
                torch.cuda.empty_cache()

        print(f"    {family}: {time.time()-t0:.1f}s")

    mean_compact = {
        k: compact_sum[k].astype(np.float32)
        for k in CORE_STATE_NAMES
    }

    save_trajectory_cache(anchor, cutoff, mean_daily, mean_compact)
    return mean_daily, mean_compact


In [19]:
required_refreshes = sorted({latest_refresh_for_anchor(a) for a in PRED_ANCHORS})
print("required refreshes:", required_refreshes)

models: dict[tuple[str, date], nn.Module] = {}
model_meta = []

for cutoff in required_refreshes:
    for family in ENABLED_FAMILIES:
        model, meta = train_or_load_model(family, cutoff)
        models[(family, cutoff)] = model
        model_meta.append(meta)

with open(REPORT_DIR / "model_meta.json", "w", encoding="utf-8") as f:
    json.dump(model_meta, f, ensure_ascii=False, indent=2)

behaviour_metric_rows = []
arch_metric_rows = []
prediction_manifest = []

for anchor in PRED_ANCHORS:
    cutoff = latest_refresh_for_anchor(anchor)
    origins = training_origins_for_cutoff(cutoff)
    strict_oot_assert(cutoff, anchor, origins)

    print("\n" + "=" * 100)
    print("PREDICT anchor=", anchor, "fit cutoff=", cutoff)

    pred_path = PRED_DIR / f"future_behaviour_archetypes_anchor_{anchor}.parquet"

    if pred_path.exists() and not FORCE_REBUILD_PREDICTION_PARQUETS:
        print("existing prediction parquet will be refreshed from trajectory cache:", pred_path.name)

    family_models = {
        family: models[(family, cutoff)]
        for family in ENABLED_FAMILIES
    }

    mean_daily, mean_compact = predict_mean_ensemble_anchor(
        family_models, anchor, cutoff
    )

    pred_stats = sufficient_from_daily(mean_daily)
    pred_arch = archetype_projection_from_sufficient(
        pred_stats,
        cutoff,
        predicted=True,
    )
    arch_frame = archetype_projection_to_frame(
        pred_arch,
        anchor,
        cutoff,
    )

    base_cols = {
        "user_id": ALL_USER_IDS,
        "anchor_date": np.full(len(ALL_USER_IDS), np.datetime64(anchor, "D")),
        "aux_fit_cutoff": np.full(len(ALL_USER_IDS), np.datetime64(cutoff, "D")),
        "aux_available": np.ones(len(ALL_USER_IDS), dtype=np.int8),
    }
    for k in CORE_STATE_NAMES:
        base_cols[f"aux_mean_{k}"] = mean_compact[k]

    pred_df = (
        pl.DataFrame(base_cols)
        .join(arch_frame, on=["user_id", "anchor_date"], how="left")
    )

    pred_df.write_parquet(
        pred_path,
        compression="zstd",
        statistics=True,
    )

    prediction_manifest.append({
        "anchor": str(anchor),
        "fit_cutoff": str(cutoff),
        "path": str(pred_path),
        "rows": pred_df.height,
        "cols": pred_df.width,
        "postny_k6_refinement_enabled": bool(anchor >= POSTNY_REFINEMENT_START),
        "k6_basis_has_completed_postny_33d": bool(
            "post_ny_33d" in PIT_ARCH[cutoff]["completed_intervals"]
        ),
    })

    print("saved:", pred_path, pred_df.shape)

    if anchor + timedelta(days=HORIZON) <= MAX_DATE:
        truth_state = build_truth_state_all_users(anchor, ALL_USER_IDS)
        split = (
            "primary" if anchor in PRIMARY_FOLDS
            else ("stress" if anchor in STRESS_FOLDS else "other")
        )

        behaviour_row = {
            "anchor": str(anchor),
            "split": split,
            "family": "mean",
            "fit_cutoff": str(cutoff),
            **evaluate_state(mean_compact, truth_state),
        }
        behaviour_metric_rows.append(behaviour_row)

        truth_stats = build_truth_future_sufficient_all_users(anchor)
        truth_arch = archetype_projection_from_sufficient(
            truth_stats,
            cutoff,
            predicted=False,
        )

        for k in [ARCH_K5, ARCH_K6]:
            arch_metric_rows.append({
                "anchor": str(anchor),
                "split": split,
                "k": k,
                "scenario": "K5" if k == ARCH_K5 else "K6_diagnostic",
                "postny_refinement_enabled": bool(anchor >= POSTNY_REFINEMENT_START),
                "basis_has_completed_postny_33d": bool(
                    "post_ny_33d" in PIT_ARCH[cutoff]["completed_intervals"]
                ),
                "fit_cutoff": str(cutoff),
                **evaluate_archetype_projection(pred_arch, truth_arch, k),
            })

        print(
            f"  METRIC mean p_any_order_logloss="
            f"{behaviour_row['p_any_order_logloss']:.5f} "
            f"order_days_MAE={behaviour_row['order_days_mae']:.4f} "
            f"cart_days_MAE={behaviour_row['cart_days_mae']:.4f}"
        )

        del truth_state, truth_stats, truth_arch

    del mean_daily, mean_compact, pred_stats, pred_arch, arch_frame, pred_df
    gc.collect()
    if CUDA_AVAILABLE:
        torch.cuda.empty_cache()

pl.DataFrame(prediction_manifest).write_csv(
    REPORT_DIR / "prediction_manifest.csv"
)

behaviour_metrics_df = (
    pl.DataFrame(behaviour_metric_rows)
    if behaviour_metric_rows else pl.DataFrame()
)
arch_metrics_df = (
    pl.DataFrame(arch_metric_rows)
    if arch_metric_rows else pl.DataFrame()
)

if behaviour_metrics_df.height:
    behaviour_metrics_df.write_csv(REPORT_DIR / "behaviour_metrics_by_anchor.csv")
    display(behaviour_metrics_df)

if arch_metrics_df.height:
    arch_metrics_df.write_csv(REPORT_DIR / "archetype_forecast_metrics_by_anchor.csv")
    display(arch_metrics_df)


required refreshes: [datetime.date(2025, 2, 13), datetime.date(2025, 5, 7), datetime.date(2025, 7, 30), datetime.date(2025, 10, 22), datetime.date(2026, 1, 14), datetime.date(2026, 2, 13)]

TRAIN gru cutoff=2025-02-13 origins=[datetime.date(2025, 1, 14)]
  block 1/1 origin=2025-01-14 X=(20000, 180, 17) Y=(20000, 30, 8) sec=0.5
  epoch 01/7 loss=3.6022 sec=1.5
  epoch 02/7 loss=2.2104 sec=1.3
  epoch 03/7 loss=1.8860 sec=1.3
  epoch 04/7 loss=1.7959 sec=1.4
  epoch 05/7 loss=1.7757 sec=1.3
  epoch 06/7 loss=1.7691 sec=1.3
  epoch 07/7 loss=1.7515 sec=1.3
saved: gru_cutoff_2025-02-13_813ed2fb5d07.pt

TRAIN tcn cutoff=2025-02-13 origins=[datetime.date(2025, 1, 14)]
  block 1/1 origin=2025-01-14 X=(20000, 180, 17) Y=(20000, 30, 8) sec=0.4
  epoch 01/7 loss=2.1199 sec=1.8
  epoch 02/7 loss=1.7904 sec=1.7
  epoch 03/7 loss=1.7609 sec=1.7
  epoch 04/7 loss=1.7688 sec=1.8
  epoch 05/7 loss=1.7515 sec=1.7
  epoch 06/7 loss=1.7455 sec=1.7
  epoch 07/7 loss=1.7380 sec=1.7
saved: tcn_cutoff_2025-0

C:\Users\Никита\AppData\Local\Temp\ipykernel_7272\1287530167.py:122: UserWarning: The given NumPy array is not writable, and PyTorch does not support non-writable tensors. This means writing to this tensor will result in undefined behavior. You may want to copy the array to protect its data or make it writable before converting it to a tensor. This type of warning will be suppressed for the rest of this program. (Triggered internally at C:\actions-runner\_work\pytorch\pytorch\torch\csrc\utils\tensor_numpy.cpp:219.)
  fb = torch.from_numpy(fc).float().to(DEVICE, non_blocking=True)


    gru: 14.4s
  inference family=tcn
    tcn: 13.1s
saved: D:\baboteek_holmotech\artifacts\prepare_features\_cache\predictions\future_behaviour_archetypes_anchor_2025-02-13.parquet (250000, 70)
  METRIC mean p_any_order_logloss=0.52313 order_days_MAE=3.1068 cart_days_MAE=4.5898

PREDICT anchor= 2025-03-26 fit cutoff= 2025-02-13
  inference family=gru
    gru: 14.9s
  inference family=tcn
    tcn: 13.4s
saved: D:\baboteek_holmotech\artifacts\prepare_features\_cache\predictions\future_behaviour_archetypes_anchor_2025-03-26.parquet (250000, 70)
  METRIC mean p_any_order_logloss=0.54462 order_days_MAE=4.5762 cart_days_MAE=5.4254

PREDICT anchor= 2025-05-07 fit cutoff= 2025-05-07
  inference family=gru
    gru: 15.6s
  inference family=tcn
    tcn: 14.0s
saved: D:\baboteek_holmotech\artifacts\prepare_features\_cache\predictions\future_behaviour_archetypes_anchor_2025-05-07.parquet (250000, 70)
  METRIC mean p_any_order_logloss=0.45098 order_days_MAE=2.0567 cart_days_MAE=3.3745

PREDICT anc

anchor,split,family,fit_cutoff,p_any_order_logloss,p_any_order_brier,p_any_active_logloss,p_any_active_brier,order_days_mae,active_days_mae,search_days_mae,cat_days_mae,cart_days_mae,to_ord_rmsle,to_cart_rmsle,searches_rmsle
str,str,str,str,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2025-02-13""","""other""","""mean""","""2025-02-13""",0.523133,0.172266,0.311381,0.102409,3.106768,4.718945,5.216663,6.217513,4.589813,0.964521,1.196059,1.2996
"""2025-03-26""","""other""","""mean""","""2025-02-13""",0.544622,0.180592,0.389197,0.12745,4.576165,5.533929,5.782944,7.890078,5.425366,1.222919,1.303949,1.451796
"""2025-05-07""","""other""","""mean""","""2025-05-07""",0.450981,0.145986,0.238503,0.073612,2.056662,3.872009,4.492964,4.672282,3.374524,0.697622,0.914732,1.076197
"""2025-06-18""","""other""","""mean""","""2025-05-07""",0.483891,0.158665,0.280666,0.09169,3.419328,6.372195,7.498142,7.143943,5.309601,0.937606,1.208906,1.413287
"""2025-07-30""","""primary""","""mean""","""2025-07-30""",0.462873,0.150953,0.221606,0.068773,2.209436,3.643026,4.118669,5.335122,3.375607,0.701838,0.868496,0.965604
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2025-10-22""","""primary""","""mean""","""2025-10-22""",0.479218,0.157467,0.142224,0.041573,2.299333,3.977007,4.551373,5.870229,4.168625,0.705109,0.974021,0.977698
"""2025-11-12""","""other""","""mean""","""2025-10-22""",0.490648,0.162336,0.066924,0.015989,2.533502,4.322762,5.104961,7.026944,4.9145,0.736557,1.050627,0.941958
"""2025-12-03""","""stress""","""mean""","""2025-10-22""",0.495544,0.164806,0.06702,0.014565,2.572344,4.842894,5.855207,7.389754,5.381839,0.739924,1.086681,1.01246


anchor,split,k,scenario,postny_refinement_enabled,basis_has_completed_postny_33d,fit_cutoff,rows,truth_weight_eligible_share,pred_weight_eligible_share,mean_weight_cosine,median_weight_cosine,weight_mae,dominant_agreement,rank_mae
str,str,i64,str,bool,bool,str,i64,f64,f64,f64,f64,f64,f64,f64
"""2025-02-13""","""other""",5,"""K5""",false,false,"""2025-02-13""",162543,0.650172,1.0,0.565018,0.593233,0.213423,0.313812,0.297302
"""2025-02-13""","""other""",6,"""K6_diagnostic""",false,false,"""2025-02-13""",162543,0.650172,1.0,0.541583,0.564317,0.182624,0.256997,0.297302
"""2025-03-26""","""other""",5,"""K5""",false,false,"""2025-02-13""",166365,0.66546,1.0,0.51191,0.509577,0.228319,0.266637,0.307657
"""2025-03-26""","""other""",6,"""K6_diagnostic""",false,false,"""2025-02-13""",166365,0.66546,1.0,0.510734,0.514891,0.189565,0.235645,0.307657
"""2025-05-07""","""other""",5,"""K5""",false,false,"""2025-05-07""",170209,0.680836,0.85228,0.659571,0.720537,0.183053,0.413662,0.265367
…,…,…,…,…,…,…,…,…,…,…,…,…,…,…
"""2025-12-03""","""stress""",6,"""K6_diagnostic""",false,false,"""2025-10-22""",222416,0.889664,1.0,0.67902,0.731593,0.1461,0.402224,0.237559
"""2025-12-15""","""stress""",5,"""K5""",false,false,"""2025-10-22""",217740,0.87096,0.99966,0.707966,0.777199,0.165262,0.468715,0.23873
"""2025-12-15""","""stress""",6,"""K6_diagnostic""",false,false,"""2025-10-22""",217740,0.87096,0.99966,0.676157,0.727904,0.146817,0.400055,0.23873


## Финальный экспорт

In [20]:
from __future__ import annotations

from datetime import date, timedelta
from pathlib import Path
import gc
import hashlib
import json
import pickle
import zipfile

import numpy as np
import polars as pl
from IPython.display import display

try:
    from archetypes import AA
except Exception as e:
    raise RuntimeError(
        "Не установлен archetypes==0.12.2. Выполни: uv add archetypes==0.12.2"
    ) from e

assert hasattr(AA, "transform"), "Нужна версия archetypes с AA.transform"
SEED = 42
np.random.seed(SEED)

print("Polars:", pl.__version__)
print("AA.transform:", hasattr(AA, "transform"))
print("V4 is CPU-only: no torch/model loading/inference.")


Polars: 1.43.2
AA.transform: True
V4 is CPU-only: no torch/model loading/inference.


In [21]:
PROJECT_DIR = Path.cwd()

def _first_existing(*paths: Path) -> Path:
    for p in paths:
        if p.exists():
            return p
    return paths[0]

TRAIN_PATH = _first_existing(
    PROJECT_DIR / "data" / "train.parquet",
    PROJECT_DIR / "train.parquet",
)
SAMPLE_PATH = _first_existing(
    PROJECT_DIR / "data" / "sample_submit.csv",
    PROJECT_DIR / "data" / "sample_submission.csv",
    PROJECT_DIR / "sample_submit.csv",
    PROJECT_DIR / "sample_submission.csv",
)

FINAL_FEATURE_ROOT = PROJECT_DIR / "artifacts" / "prepare_features"
SOURCE_V2_ROOT = FINAL_FEATURE_ROOT / "_cache"
SOURCE_V2_TRAJECTORY_DIR = SOURCE_V2_ROOT / "mean_daily_trajectories"
SOURCE_V2_ARCH_DIR = SOURCE_V2_ROOT / "archetype_pit"
SOURCE_V2_CACHE_DIR = SOURCE_V2_ROOT / "cache"


ARTIFACT_ROOT = FINAL_FEATURE_ROOT
PRED_DIR = ARTIFACT_ROOT / "predictions_final"
REPORT_DIR = ARTIFACT_ROOT / "reports"
for p in [ARTIFACT_ROOT, PRED_DIR, REPORT_DIR]:
    p.mkdir(parents=True, exist_ok=True)

for required in [TRAIN_PATH, SAMPLE_PATH, SOURCE_V2_ROOT, SOURCE_V2_TRAJECTORY_DIR, SOURCE_V2_ARCH_DIR, SOURCE_V2_CACHE_DIR]:
    if not required.exists():
        raise FileNotFoundError(f"Не найден required source: {required}")

HORIZON = 30
EXPECTED_MIN_DATE = date(2025, 1, 1)
FINAL_ANCHOR = date(2026, 2, 13)
EXPECTED_USERS = 250_000
CANONICAL_ANCHORS = [
    date(2025, 2, 13), date(2025, 3, 26), date(2025, 5, 7),
    date(2025, 6, 18), date(2025, 7, 30), date(2025, 9, 10),
    date(2025, 10, 22), date(2025, 11, 12), date(2025, 12, 3),
    date(2025, 12, 15), date(2026, 1, 14),
]
PRED_ANCHORS = CANONICAL_ANCHORS + [FINAL_ANCHOR]
FULL_REFRESH_ANCHORS = [date(2025, 2, 13), date(2025, 5, 7), date(2025, 7, 30), date(2025, 10, 22), date(2026, 1, 14), FINAL_ANCHOR]

ARCH_K5 = 5
ARCH_K6 = 6
POSTNY_REFINEMENT_START = date(2026, 1, 12)
NY_REGIME_START = date(2025, 12, 15)
NY_REGIME_END = date(2026, 1, 11)
CORE_ARCH_FEATURES = [
    "search_day_propensity", "cat_day_propensity", "searches_per_active_day",
    "cart_day_propensity", "carts_per_cart_day", "order_vs_cart_balance",
    "order_day_propensity", "orders_per_order_day",
]
CORRECTED_INTERVAL_SPECS = [
    (date(2025, 1, 12), date(2025, 2, 13), "ordinary_0"),
    (date(2025, 2, 14), date(2025, 3, 26), "ordinary_1"),
    (date(2025, 3, 27), date(2025, 5, 7), "ordinary_2"),
    (date(2025, 5, 8), date(2025, 6, 18), "ordinary_3"),
    (date(2025, 6, 19), date(2025, 7, 30), "ordinary_4"),
    (date(2025, 7, 31), date(2025, 9, 10), "ordinary_5"),
    (date(2025, 9, 11), date(2025, 10, 22), "ordinary_6"),
    (date(2025, 10, 23), date(2025, 11, 12), "ordinary_7"),
    (date(2025, 11, 13), date(2025, 12, 14), "pre_ny_32d"),
    (date(2026, 1, 12), date(2026, 2, 13), "post_ny_33d"),
]
PIT_AA_TRANSFORM_CHUNK = 100_000
MIN_CURRENT_WEIGHT_ACTIVE_DAYS = 3
K5_CANONICAL_FROM_V2_PIT = np.asarray([3, 2, 4, 1, 0], dtype=int)
DAILY_PRED_KEYS = ["p_order", "p_active", "p_search", "p_cat", "p_cart", "e_orders", "e_carts", "e_searches"]
CORE_STATE_NAMES = [
    "p_any_order", "p_any_active", "p_inactive_30d", "p_active_no_order_30d",
    "future_order_days_30d", "future_to_ord_30d", "future_active_days_30d", "orders_per_order_day",
    "future_search_days_30d", "future_cat_days_30d", "future_cart_days_30d", "future_searches_30d", "future_to_cart_30d",
    "searches_per_active_day", "carts_per_cart_day",
    "order_days_w1", "order_days_w2", "order_days_w3", "order_days_w4",
    "active_days_w1", "active_days_w2", "active_days_w3", "active_days_w4",
    "order_days_trend", "active_days_trend", "weekend_order_share",
]
SUFFICIENT_NAMES = ["active_days", "search_days", "cat_days", "cart_days", "order_days", "searches_sum", "to_cart_sum", "to_ord_sum"]

print("SOURCE V2:", SOURCE_V2_ROOT.resolve())
print("OUTPUT V4:", ARTIFACT_ROOT.resolve())


SOURCE V2: D:\baboteek_holmotech\artifacts\prepare_features\_cache
OUTPUT V4: D:\baboteek_holmotech\artifacts\prepare_features


In [22]:
REQUIRED_TRAIN_COLS = ["event_date", "user_id", "search", "cat", "to_cart", "to_ord", "searches"]
sample = pl.read_csv(SAMPLE_PATH)
if "user_id" not in sample.columns:
    raise RuntimeError("sample submission не содержит user_id")
all_users = sample.select(pl.col("user_id").cast(pl.Int64)).unique(maintain_order=True)
ALL_USER_IDS = all_users["user_id"].to_numpy()
if len(ALL_USER_IDS) != EXPECTED_USERS:
    raise RuntimeError(f"Ожидалось {EXPECTED_USERS} users, найдено {len(ALL_USER_IDS)}")
if len(np.unique(ALL_USER_IDS)) != len(ALL_USER_IDS):
    raise RuntimeError("duplicate user_id в sample submission")

schema = pl.read_parquet_schema(TRAIN_PATH)
missing = [c for c in REQUIRED_TRAIN_COLS if c not in schema]
if missing:
    raise RuntimeError(f"train.parquet missing columns: {missing}")

daily = (
    pl.scan_parquet(TRAIN_PATH)
    .select(REQUIRED_TRAIN_COLS)
    .with_columns([
        pl.col("event_date").cast(pl.Date), pl.col("user_id").cast(pl.Int64),
        pl.col("search").fill_null(0).cast(pl.Float32), pl.col("cat").fill_null(0).cast(pl.Float32),
        pl.col("to_cart").fill_null(0).cast(pl.Float32), pl.col("to_ord").fill_null(0).cast(pl.Float32),
        pl.col("searches").fill_null(0).cast(pl.Float32),
    ])
    .collect(engine="streaming")
    .sort(["event_date", "user_id"])
)

bounds = daily.select([pl.col("event_date").min().alias("min_date"), pl.col("event_date").max().alias("max_date"), pl.len().alias("rows")])
MIN_DATE = bounds["min_date"][0]
MAX_DATE = bounds["max_date"][0]
if MAX_DATE != FINAL_ANCHOR:
    raise RuntimeError(f"Unexpected max date {MAX_DATE}; expected {FINAL_ANCHOR}")
if MIN_DATE > EXPECTED_MIN_DATE:
    raise RuntimeError(f"Unexpected min date {MIN_DATE}; expected <= {EXPECTED_MIN_DATE}")

USER_ORDER_HASH = hashlib.md5(np.asarray(ALL_USER_IDS, dtype=np.int64).tobytes()).hexdigest()[:12]
v2_ids_path = SOURCE_V2_ROOT / "canonical_user_ids.npy"
if v2_ids_path.exists() and not np.array_equal(np.load(v2_ids_path), ALL_USER_IDS):
    raise RuntimeError("V2 canonical_user_ids.npy не совпадает с sample submission")
np.save(ARTIFACT_ROOT / "canonical_user_ids.npy", ALL_USER_IDS)
print(bounds)
print("users:", len(ALL_USER_IDS), "user order hash:", USER_ORDER_HASH)


shape: (1, 3)
┌────────────┬────────────┬──────────┐
│ min_date   ┆ max_date   ┆ rows     │
│ ---        ┆ ---        ┆ ---      │
│ date       ┆ date       ┆ u32      │
╞════════════╪════════════╪══════════╡
│ 2025-01-01 ┆ 2026-02-13 ┆ 30631006 │
└────────────┴────────────┴──────────┘
users: 250000 user order hash: d900373f804d


In [23]:
def latest_refresh_for_anchor(anchor: date) -> date:
    valid = [r for r in FULL_REFRESH_ANCHORS if r <= anchor]
    if not valid:
        raise RuntimeError(f"No refresh <= anchor {anchor}")
    return max(valid)

def completed_arch_intervals(cutoff: date):
    return [(start, end, label) for start, end, label in CORRECTED_INTERVAL_SPECS if end <= cutoff]

rows=[]
for anchor in PRED_ANCHORS:
    cutoff=latest_refresh_for_anchor(anchor)
    comp=completed_arch_intervals(cutoff)
    rows.append({
        "prediction_anchor": str(anchor), "fit_cutoff": str(cutoff), "n_completed_arch_intervals": len(comp),
        "latest_arch_interval": comp[-1][2] if comp else None,
        "basis_has_completed_postny_33d": any(x[2]=="post_ny_33d" for x in comp),
    })
refresh_df=pl.DataFrame(rows)
refresh_df.write_csv(REPORT_DIR / "strict_oot_refresh_plan_v4_final.csv")
display(refresh_df)


prediction_anchor,fit_cutoff,n_completed_arch_intervals,latest_arch_interval,basis_has_completed_postny_33d
str,str,i64,str,bool
"""2025-02-13""","""2025-02-13""",1,"""ordinary_0""",false
"""2025-03-26""","""2025-02-13""",1,"""ordinary_0""",false
"""2025-05-07""","""2025-05-07""",3,"""ordinary_2""",false
"""2025-06-18""","""2025-05-07""",3,"""ordinary_2""",false
"""2025-07-30""","""2025-07-30""",5,"""ordinary_4""",false
…,…,…,…,…
"""2025-11-12""","""2025-10-22""",7,"""ordinary_6""",false
"""2025-12-03""","""2025-10-22""",7,"""ordinary_6""",false
"""2025-12-15""","""2025-10-22""",7,"""ordinary_6""",false


In [24]:
def load_latest_priors(cutoff: date) -> dict:
    completed = completed_arch_intervals(cutoff)
    if not completed:
        raise RuntimeError(f"No completed archetype interval for cutoff={cutoff}")
    _, end, _ = completed[-1]
    p = SOURCE_V2_CACHE_DIR / f"arch_prior_interval_end_{end}.json"
    if not p.exists():
        raise FileNotFoundError(f"Не найден V2 prior cache: {p}")
    with open(p, "r", encoding="utf-8") as f:
        return json.load(f)

def load_v2_pit_arch(cutoffs: list[date]) -> dict:
    result={}
    for cutoff in sorted(cutoffs):
        completed=completed_arch_intervals(cutoff)
        result[cutoff]={"completed_intervals":[x[2] for x in completed], "latest_priors":load_latest_priors(cutoff), "models":{}}
        for k in [ARCH_K5, ARCH_K6]:
            p=SOURCE_V2_ARCH_DIR / f"aa_k{k}_cutoff_{cutoff}.pkl"
            if not p.exists():
                raise FileNotFoundError(f"Не найден готовый V2 PIT basis: {p}")
            with open(p,"rb") as f:
                payload=pickle.load(f)
            result[cutoff]["models"][k]={
                "model":payload["model"], "perm":np.asarray(payload["perm_to_aligned"],dtype=int),
                "aligned_profiles":np.asarray(payload["aligned_profiles"],dtype=np.float64),
            }
    return result

REQUIRED_REFRESHES=sorted(set(latest_refresh_for_anchor(a) for a in PRED_ANCHORS))
PIT_ARCH=load_v2_pit_arch(REQUIRED_REFRESHES)
print("loaded PIT cutoffs:", REQUIRED_REFRESHES)


loaded PIT cutoffs: [datetime.date(2025, 2, 13), datetime.date(2025, 5, 7), datetime.date(2025, 7, 30), datetime.date(2025, 10, 22), datetime.date(2026, 1, 14), datetime.date(2026, 2, 13)]


In [25]:
def v2_trajectory_cache_path(anchor: date) -> Path:
    return SOURCE_V2_TRAJECTORY_DIR / f"mean_daily_anchor_{anchor}.npz"

def load_v2_mean_trajectory(anchor: date, cutoff: date):
    p=v2_trajectory_cache_path(anchor)
    if not p.exists():
        raise FileNotFoundError(f"Не найден V2 mean trajectory cache: {p}")
    z=np.load(p,allow_pickle=False)
    if "anchor" in z and str(z["anchor"].item()) != str(anchor):
        raise RuntimeError(f"anchor mismatch in {p}")
    if "cutoff" in z and str(z["cutoff"].item()) != str(cutoff):
        raise RuntimeError(f"cutoff mismatch in {p}")
    if "user_order_hash" in z and str(z["user_order_hash"].item()) != USER_ORDER_HASH:
        raise RuntimeError(f"user order hash mismatch in {p}")
    mean_daily={k:z["daily__"+k].astype(np.float32) for k in DAILY_PRED_KEYS}
    mean_daily["p_any_order"]=z["head__p_any_order"].astype(np.float32)
    mean_daily["p_any_active"]=z["head__p_any_active"].astype(np.float32)
    mean_compact={k:z["compact__"+k].astype(np.float32) for k in CORE_STATE_NAMES}
    return mean_daily, mean_compact

inv=[]
for a in PRED_ANCHORS:
    p=v2_trajectory_cache_path(a)
    inv.append({"anchor":str(a),"exists":p.exists(),"size_mb":round(p.stat().st_size/1024**2,2) if p.exists() else None})
inventory_df=pl.DataFrame(inv)
inventory_df.write_csv(REPORT_DIR / "source_v2_mean_cache_inventory_v4_final.csv")
display(inventory_df)
if not inventory_df["exists"].all():
    raise RuntimeError("Не все V2 mean trajectory cache найдены")


anchor,exists,size_mb
str,bool,f64
"""2025-02-13""",true,90.92
"""2025-03-26""",true,93.57
"""2025-05-07""",true,104.45
"""2025-06-18""",true,102.6
"""2025-07-30""",true,111.06
…,…,…
"""2025-11-12""",true,113.5
"""2025-12-03""",true,114.29
"""2025-12-15""",true,117.39


In [26]:
def beta_posterior(x,n,a,b):
    x=np.asarray(x,dtype=np.float64); n=np.asarray(n,dtype=np.float64)
    valid=np.isfinite(x)&np.isfinite(n)&(n>0)&(x>=0); x_safe=np.clip(x,0,np.maximum(n,0))
    mean=np.full(len(x),a/(a+b),dtype=np.float64); conf=np.zeros(len(x),dtype=np.float64)
    mean[valid]=(x_safe[valid]+a)/(n[valid]+a+b); conf[valid]=n[valid]/(n[valid]+a+b)
    return mean,conf

def gamma_poisson_posterior(count,exposure,a,b):
    x=np.asarray(count,dtype=np.float64); e=np.asarray(exposure,dtype=np.float64)
    valid=np.isfinite(x)&np.isfinite(e)&(e>0)&(x>=0)
    mean=np.full(len(x),a/b,dtype=np.float64); conf=np.zeros(len(x),dtype=np.float64)
    mean[valid]=(x[valid]+a)/(e[valid]+b); conf[valid]=e[valid]/(e[valid]+b)
    return mean,conf

def coords_from_sufficient(stats,priors):
    active=np.asarray(stats["active_days"],dtype=np.float64); search_days=np.asarray(stats["search_days"],dtype=np.float64)
    cat_days=np.asarray(stats["cat_days"],dtype=np.float64); cart_days=np.asarray(stats["cart_days"],dtype=np.float64)
    order_days=np.asarray(stats["order_days"],dtype=np.float64); searches=np.asarray(stats["searches_sum"],dtype=np.float64)
    carts=np.asarray(stats["to_cart_sum"],dtype=np.float64); orders=np.asarray(stats["to_ord_sum"],dtype=np.float64)
    beta_inputs={
        "search_day_propensity":(search_days,active),"cat_day_propensity":(cat_days,active),"cart_day_propensity":(cart_days,active),
        "order_day_propensity":(order_days,active),"order_vs_cart_balance":(orders,orders+carts),
    }
    gamma_inputs={"searches_per_active_day":(searches,active),"carts_per_cart_day":(carts,cart_days),"orders_per_order_day":(orders,order_days)}
    coords={}; confs=[]
    for name,(x,n) in beta_inputs.items():
        p=priors["beta"][name]; mean,conf=beta_posterior(x,n,float(p["a"]),float(p["b"])); coords[name]=mean; confs.append(conf)
    for name,(x,e) in gamma_inputs.items():
        p=priors["gamma"][name]; mean,conf=gamma_poisson_posterior(x,e,float(p["a"]),float(p["b"])); coords[name]=mean; confs.append(conf)
    eb_conf=np.nanmean(np.column_stack(confs),axis=1); active_evidence=np.minimum(1.0,active/10.0)
    measurement_conf=np.clip(0.5*eb_conf+0.5*active_evidence,0,1)
    return coords,measurement_conf

def masked_percentile_values(values,reference_mask):
    x=np.asarray(values,dtype=np.float64); mask=np.asarray(reference_mask,dtype=bool)&np.isfinite(x); ref=x[mask]
    if len(ref)<2: return np.full(len(x),0.5,dtype=np.float32)
    ref=np.sort(ref); left=np.searchsorted(ref,x,side="left"); right=np.searchsorted(ref,x,side="right")
    mid=0.5*(left+right-1); return np.clip(mid/max(len(ref)-1,1),0,1).astype(np.float32)

def rank_core_population(coords,reference_mask):
    return np.column_stack([masked_percentile_values(coords[name],reference_mask) for name in CORE_ARCH_FEATURES]).astype(np.float64,copy=False)

def transform_aa_aligned(model,perm,X):
    parts=[]
    for s in range(0,len(X),PIT_AA_TRANSFORM_CHUNK):
        parts.append(np.asarray(model.transform(X[s:s+PIT_AA_TRANSFORM_CHUNK]),dtype=np.float64)[:,perm])
    W=np.vstack(parts); W=np.clip(W,0,None); W=W/np.maximum(W.sum(axis=1,keepdims=True),1e-12)
    return W.astype(np.float32)

def normalized_entropy(W):
    x=np.clip(np.asarray(W,dtype=np.float64),1e-12,1)
    return (-np.sum(x*np.log(x),axis=1)/np.log(x.shape[1])).astype(np.float32)

def sufficient_from_daily(d):
    return {
        "active_days":np.asarray(d["p_active"],dtype=np.float64).sum(axis=1),"search_days":np.asarray(d["p_search"],dtype=np.float64).sum(axis=1),
        "cat_days":np.asarray(d["p_cat"],dtype=np.float64).sum(axis=1),"cart_days":np.asarray(d["p_cart"],dtype=np.float64).sum(axis=1),
        "order_days":np.asarray(d["p_order"],dtype=np.float64).sum(axis=1),"searches_sum":np.asarray(d["e_searches"],dtype=np.float64).sum(axis=1),
        "to_cart_sum":np.asarray(d["e_carts"],dtype=np.float64).sum(axis=1),"to_ord_sum":np.asarray(d["e_orders"],dtype=np.float64).sum(axis=1),
    }

def project_future_all_users(stats,cutoff):
    arch=PIT_ARCH[cutoff]; coords,measurement_conf=coords_from_sufficient(stats,arch["latest_priors"])
    X_rank=rank_core_population(coords,np.ones(len(ALL_USER_IDS),dtype=bool))
    out={"measurement_confidence":measurement_conf.astype(np.float32),"X_rank":X_rank.astype(np.float32)}
    for name in CORE_ARCH_FEATURES: out[f"eb__{name}"]=np.asarray(coords[name],dtype=np.float32)
    for j,name in enumerate(CORE_ARCH_FEATURES): out[f"rank__{name}"]=X_rank[:,j].astype(np.float32)
    for k in [ARCH_K5,ARCH_K6]:
        info=arch["models"][k]; W=transform_aa_aligned(info["model"],info["perm"],X_rank)
        if k==ARCH_K5: W=W[:,K5_CANONICAL_FROM_V2_PIT]
        ent=normalized_entropy(W); out[f"k{k}_weights"]=W; out[f"k{k}_dominant"]=np.argmax(W,axis=1).astype(np.int16)
        out[f"k{k}_dominant_weight"]=np.max(W,axis=1).astype(np.float32); out[f"k{k}_entropy"]=ent
        out[f"k{k}_effective_count"]=np.exp(ent*np.log(k)).astype(np.float32)
    return out

def project_observed_current(stats,cutoff):
    arch=PIT_ARCH[cutoff]; coords,measurement_conf=coords_from_sufficient(stats,arch["latest_priors"]); active=np.asarray(stats["active_days"],dtype=np.float64)
    X_rank=rank_core_population(coords,active>0); eligible=active>=MIN_CURRENT_WEIGHT_ACTIVE_DAYS
    info=arch["models"][ARCH_K5]; W=transform_aa_aligned(info["model"],info["perm"],X_rank)[:,K5_CANONICAL_FROM_V2_PIT]
    ent=normalized_entropy(W)
    return {"eligible":eligible,"measurement_confidence":measurement_conf.astype(np.float32),"weights":W.astype(np.float32),
            "dominant":np.argmax(W,axis=1).astype(np.int16),"dominant_weight":np.max(W,axis=1).astype(np.float32),
            "entropy":ent,"effective_count":np.exp(ent*np.log(ARCH_K5)).astype(np.float32)}


In [27]:
def build_observed_sufficient_all_users(start: date,end: date):
    part=daily.filter(pl.col("event_date").is_between(start,end,closed="both"))
    agg=(part.group_by("user_id").agg([
        pl.col("event_date").n_unique().alias("active_days"),
        pl.col("event_date").filter(pl.col("search")>0).n_unique().alias("search_days"),
        pl.col("event_date").filter(pl.col("cat")>0).n_unique().alias("cat_days"),
        pl.col("event_date").filter(pl.col("to_cart")>0).n_unique().alias("cart_days"),
        pl.col("event_date").filter(pl.col("to_ord")>0).n_unique().alias("order_days"),
        pl.col("searches").sum().alias("searches_sum"),pl.col("to_cart").sum().alias("to_cart_sum"),pl.col("to_ord").sum().alias("to_ord_sum"),
    ]))
    base=all_users.join(agg,on="user_id",how="left",maintain_order="left").with_columns([pl.col(c).fill_null(0) for c in SUFFICIENT_NAMES])
    return {c:base[c].to_numpy().astype(np.float64,copy=False) for c in SUFFICIENT_NAMES}

def build_current_30d_sufficient_all_users(anchor: date):
    return build_observed_sufficient_all_users(max(MIN_DATE,anchor-timedelta(days=HORIZON-1)),anchor)

def build_truth_future_sufficient_all_users(anchor: date):
    end=anchor+timedelta(days=HORIZON)
    if end>MAX_DATE: raise ValueError(f"truth unavailable for {anchor}")
    return build_observed_sufficient_all_users(anchor+timedelta(days=1),end)


In [28]:
def prob_at_least_k_bernoulli(probs,k=3):
    p=np.clip(np.asarray(probs,dtype=np.float64),1e-7,1-1e-7); n=p.shape[0]
    small=np.zeros((n,k),dtype=np.float64); small[:,0]=1.0
    for t in range(p.shape[1]):
        pt=p[:,t]; nxt=np.zeros_like(small); nxt[:,0]=small[:,0]*(1-pt)
        for j in range(1,k): nxt[:,j]=small[:,j]*(1-pt)+small[:,j-1]*pt
        small=nxt
    return np.clip(1.0-small.sum(axis=1),0,1).astype(np.float32)

def raw_availability_score(mean_daily):
    p_ge3=prob_at_least_k_bernoulli(mean_daily["p_active"],k=MIN_CURRENT_WEIGHT_ACTIVE_DAYS)
    p_any=np.clip(np.asarray(mean_daily["p_any_active"],dtype=np.float64),0,1)
    return np.minimum(p_ge3,p_any).astype(np.float32)


In [29]:
def _row_cosine(A,B):
    A=np.asarray(A,dtype=np.float64); B=np.asarray(B,dtype=np.float64); num=np.sum(A*B,axis=1); den=np.linalg.norm(A,axis=1)*np.linalg.norm(B,axis=1)
    return np.divide(num,den,out=np.zeros_like(num),where=den>1e-12)

def build_final_anchor_frame(anchor,cutoff,mean_daily,mean_compact,future_proj,current_proj):
    raw_avail=raw_availability_score(mean_daily); quality=np.clip(raw_avail*future_proj["measurement_confidence"],0,1).astype(np.float32)
    Wf=future_proj["k5_weights"].astype(np.float32); Wc=current_proj["weights"].astype(np.float32); current_eligible=np.asarray(current_proj["eligible"],dtype=bool)
    change_cos=_row_cosine(Wf,Wc).astype(np.float32); change_l1=np.abs(Wf-Wc).sum(axis=1).astype(np.float32)
    dominant_same=(future_proj["k5_dominant"]==current_proj["dominant"]).astype(np.float32)
    cols={
        "user_id":ALL_USER_IDS,"anchor_date":np.full(len(ALL_USER_IDS),np.datetime64(anchor,"D")),
        "aux_fit_cutoff":np.full(len(ALL_USER_IDS),np.datetime64(cutoff,"D")),"arch_fit_cutoff":np.full(len(ALL_USER_IDS),np.datetime64(cutoff,"D")),
        "aux_available":np.ones(len(ALL_USER_IDS),dtype=np.int8),
        "future_arch_availability_score":raw_avail,"future_arch_measurement_confidence":future_proj["measurement_confidence"].astype(np.float32),
        "future_arch_quality_score":quality,"future_arch_rank_reference_all_users":np.ones(len(ALL_USER_IDS),dtype=np.int8),
        "k5_canonical_v31_order":np.ones(len(ALL_USER_IDS),dtype=np.int8),
        "current_arch5_weight_eligible":current_eligible.astype(np.int8),"current_arch5_measurement_confidence":current_proj["measurement_confidence"].astype(np.float32),
        "postny_k6_refinement_enabled":np.full(len(ALL_USER_IDS),int(anchor>=POSTNY_REFINEMENT_START),dtype=np.int8),
        "k6_basis_has_completed_postny_33d":np.full(len(ALL_USER_IDS),int("post_ny_33d" in PIT_ARCH[cutoff]["completed_intervals"]),dtype=np.int8),
    }
    for name in CORE_STATE_NAMES: cols[f"aux_mean_{name}"]=mean_compact[name].astype(np.float32)
    for name in CORE_ARCH_FEATURES:
        cols[f"future_arch_eb__{name}"]=future_proj[f"eb__{name}"].astype(np.float32); cols[f"future_arch_rank__{name}"]=future_proj[f"rank__{name}"].astype(np.float32)
    for j in range(ARCH_K5): cols[f"future_arch5_w_{j}"]=Wf[:,j]
    cols["future_arch5_dominant"]=future_proj["k5_dominant"]; cols["future_arch5_dominant_weight"]=future_proj["k5_dominant_weight"]
    cols["future_arch5_entropy"]=future_proj["k5_entropy"]; cols["future_arch5_effective_count"]=future_proj["k5_effective_count"]
    W6=future_proj["k6_weights"].astype(np.float32)
    for j in range(ARCH_K6): cols[f"future_arch6_w_{j}"]=W6[:,j]
    cols["future_arch6_dominant"]=future_proj["k6_dominant"]; cols["future_arch6_dominant_weight"]=future_proj["k6_dominant_weight"]
    cols["future_arch6_entropy"]=future_proj["k6_entropy"]; cols["future_arch6_effective_count"]=future_proj["k6_effective_count"]

    for j in range(ARCH_K5):
        cur=Wc[:,j].copy(); delta=(Wf[:,j]-Wc[:,j]).astype(np.float32); cur[~current_eligible]=np.nan; delta[~current_eligible]=np.nan
        cols[f"current_arch5_w_{j}"]=cur.astype(np.float32); cols[f"future_arch5_delta_w_{j}"]=delta
    cur_dom=current_proj["dominant"].astype(np.float32); cur_dw=current_proj["dominant_weight"].astype(np.float32)
    cur_ent=current_proj["entropy"].astype(np.float32); cur_eff=current_proj["effective_count"].astype(np.float32)
    for arr in [cur_dom,cur_dw,cur_ent,cur_eff,change_cos,change_l1,dominant_same]: arr[~current_eligible]=np.nan
    cols["current_arch5_dominant"]=cur_dom; cols["current_arch5_dominant_weight"]=cur_dw; cols["current_arch5_entropy"]=cur_ent; cols["current_arch5_effective_count"]=cur_eff
    cols["future_vs_current_arch5_cosine"]=change_cos; cols["future_vs_current_arch5_l1"]=change_l1; cols["future_vs_current_arch5_dominant_same"]=dominant_same
    df=pl.DataFrame(cols)
    nan_cols=[*[f"current_arch5_w_{j}" for j in range(ARCH_K5)],*[f"future_arch5_delta_w_{j}" for j in range(ARCH_K5)],
              "current_arch5_dominant","current_arch5_dominant_weight","current_arch5_entropy","current_arch5_effective_count",
              "future_vs_current_arch5_cosine","future_vs_current_arch5_l1","future_vs_current_arch5_dominant_same"]
    return df.with_columns([pl.when(pl.col(c).is_nan()).then(None).otherwise(pl.col(c)).alias(c) for c in nan_cols])


In [30]:
manifest_rows=[]
for anchor in PRED_ANCHORS:
    cutoff=latest_refresh_for_anchor(anchor)
    print("\n"+"="*90); print("V4 FINAL anchor=",anchor,"cutoff=",cutoff)
    mean_daily,mean_compact=load_v2_mean_trajectory(anchor,cutoff)
    future_proj=project_future_all_users(sufficient_from_daily(mean_daily),cutoff)
    current_proj=project_observed_current(build_current_30d_sufficient_all_users(anchor),cutoff)
    df=build_final_anchor_frame(anchor,cutoff,mean_daily,mean_compact,future_proj,current_proj)
    fcols=[f"future_arch5_w_{j}" for j in range(ARCH_K5)]; assert all(df[c].null_count()==0 for c in fcols)
    assert np.allclose(df.select(pl.sum_horizontal(fcols).alias("s"))["s"].to_numpy(),1.0,atol=2e-4)
    ccols=[f"current_arch5_w_{j}" for j in range(ARCH_K5)]; elig=df["current_arch5_weight_eligible"].to_numpy().astype(bool); nnull=int((~elig).sum())
    assert all(df[c].null_count()==nnull for c in ccols)
    if elig.any():
        cs=df.filter(pl.col("current_arch5_weight_eligible")==1).select(pl.sum_horizontal(ccols).alias("s"))["s"].to_numpy(); assert np.allclose(cs,1.0,atol=2e-4)
    p=PRED_DIR / f"future_behaviour_archetypes_v4_final_anchor_{anchor}.parquet"; df.write_parquet(p,compression="zstd",statistics=True)
    manifest_rows.append({"anchor":str(anchor),"fit_cutoff":str(cutoff),"rows":df.height,"cols":df.width,"future_k5_null_share":0.0,
                          "future_availability_score_mean":float(df["future_arch_availability_score"].mean()),"future_quality_score_mean":float(df["future_arch_quality_score"].mean()),
                          "current_k5_eligible_share":float(df["current_arch5_weight_eligible"].mean()),"postny_k6_refinement_enabled":bool(anchor>=POSTNY_REFINEMENT_START),
                          "k6_basis_has_completed_postny_33d":bool("post_ny_33d" in PIT_ARCH[cutoff]["completed_intervals"]),"path":str(p)})
    del mean_daily,mean_compact,future_proj,current_proj,df; gc.collect()
manifest_df=pl.DataFrame(manifest_rows); manifest_df.write_csv(REPORT_DIR / "prediction_manifest_v4_final.csv"); display(manifest_df)



V4 FINAL anchor= 2025-02-13 cutoff= 2025-02-13

V4 FINAL anchor= 2025-03-26 cutoff= 2025-02-13

V4 FINAL anchor= 2025-05-07 cutoff= 2025-05-07

V4 FINAL anchor= 2025-06-18 cutoff= 2025-05-07

V4 FINAL anchor= 2025-07-30 cutoff= 2025-07-30

V4 FINAL anchor= 2025-09-10 cutoff= 2025-07-30

V4 FINAL anchor= 2025-10-22 cutoff= 2025-10-22

V4 FINAL anchor= 2025-11-12 cutoff= 2025-10-22

V4 FINAL anchor= 2025-12-03 cutoff= 2025-10-22

V4 FINAL anchor= 2025-12-15 cutoff= 2025-10-22

V4 FINAL anchor= 2026-01-14 cutoff= 2026-01-14

V4 FINAL anchor= 2026-02-13 cutoff= 2026-02-13


anchor,fit_cutoff,rows,cols,future_k5_null_share,future_availability_score_mean,future_quality_score_mean,current_k5_eligible_share,postny_k6_refinement_enabled,k6_basis_has_completed_postny_33d,path
str,str,i64,i64,f64,f64,f64,f64,bool,bool,str
"""2025-02-13""","""2025-02-13""",250000,92,0.0,0.792414,0.587263,0.625704,false,false,"""D:\baboteek_holmotech\artifact…"
"""2025-03-26""","""2025-02-13""",250000,92,0.0,0.720326,0.56835,0.656024,false,false,"""D:\baboteek_holmotech\artifact…"
"""2025-05-07""","""2025-05-07""",250000,92,0.0,0.848388,0.596835,0.668808,false,false,"""D:\baboteek_holmotech\artifact…"
"""2025-06-18""","""2025-05-07""",250000,92,0.0,0.912097,0.724184,0.687244,false,false,"""D:\baboteek_holmotech\artifact…"
"""2025-07-30""","""2025-07-30""",250000,92,0.0,0.833244,0.589956,0.704056,false,false,"""D:\baboteek_holmotech\artifact…"
…,…,…,…,…,…,…,…,…,…,…
"""2025-11-12""","""2025-10-22""",250000,92,0.0,0.957023,0.754196,0.820276,false,false,"""D:\baboteek_holmotech\artifact…"
"""2025-12-03""","""2025-10-22""",250000,92,0.0,0.984725,0.793484,0.86116,false,false,"""D:\baboteek_holmotech\artifact…"
"""2025-12-15""","""2025-10-22""",250000,92,0.0,0.975888,0.756737,0.882604,false,false,"""D:\baboteek_holmotech\artifact…"


In [31]:
def _metrics_three(Wp, Wc, Wt):
    cos_p = _row_cosine(Wp, Wt)
    cos_c = _row_cosine(Wc, Wt)
    return {
        "rows": int(len(Wt)),
        "persistence_mean_cosine": float(np.mean(cos_c)),
        "temporal_mean_cosine": float(np.mean(cos_p)),
        "mean_cosine_lift": float(np.mean(cos_p) - np.mean(cos_c)),
        "persistence_total_l1": float(np.mean(np.abs(Wc - Wt).sum(axis=1))),
        "temporal_total_l1": float(np.mean(np.abs(Wp - Wt).sum(axis=1))),
        "total_l1_improvement": float(np.mean(np.abs(Wc - Wt).sum(axis=1)) - np.mean(np.abs(Wp - Wt).sum(axis=1))),
        "persistence_dominant_agreement": float(np.mean(np.argmax(Wc, axis=1) == np.argmax(Wt, axis=1))),
        "temporal_dominant_agreement": float(np.mean(np.argmax(Wp, axis=1) == np.argmax(Wt, axis=1))),
        "dominant_agreement_lift": float(np.mean(np.argmax(Wp, axis=1) == np.argmax(Wt, axis=1)) - np.mean(np.argmax(Wc, axis=1) == np.argmax(Wt, axis=1))),
    }

freeze_rows=[]
for anchor in CANONICAL_ANCHORS:
    if anchor + timedelta(days=HORIZON) > MAX_DATE:
        continue
    cutoff=latest_refresh_for_anchor(anchor)
    p=PRED_DIR / f"future_behaviour_archetypes_v4_final_anchor_{anchor}.parquet"
    pred=pl.read_parquet(p)
    truth_proj=project_observed_current(build_truth_future_sufficient_all_users(anchor),cutoff)
    current_eligible=pred["current_arch5_weight_eligible"].to_numpy().astype(bool)
    truth_eligible=np.asarray(truth_proj["eligible"],dtype=bool)
    mask=current_eligible & truth_eligible
    Wp=np.column_stack([pred[f"future_arch5_w_{j}"].to_numpy() for j in range(ARCH_K5)])[mask].astype(np.float64)
    Wc=np.column_stack([pred[f"current_arch5_w_{j}"].to_numpy() for j in range(ARCH_K5)])[mask].astype(np.float64)
    Wt=truth_proj["weights"][mask].astype(np.float64)
    split="primary" if anchor in [date(2025,7,30),date(2025,9,10),date(2025,10,22),date(2026,1,14)] else ("stress" if anchor in [date(2025,12,3),date(2025,12,15)] else "other")
    freeze_rows.append({"anchor":str(anchor),"split":split,"fit_cutoff":str(cutoff),**_metrics_three(Wp,Wc,Wt)})
    del pred,truth_proj,Wp,Wc,Wt; gc.collect()

freeze_df=pl.DataFrame(freeze_rows)
freeze_df.write_csv(REPORT_DIR / "k5_persistence_vs_temporal_freeze_check_v4_final.csv")
display(freeze_df)

freeze_summary=(freeze_df.group_by("split").agg([
    pl.len().alias("anchors"),
    pl.col("mean_cosine_lift").mean(),
    pl.col("total_l1_improvement").mean(),
    pl.col("dominant_agreement_lift").mean(),
]).sort("split"))
freeze_summary.write_csv(REPORT_DIR / "k5_persistence_vs_temporal_freeze_summary_v4_final.csv")
display(freeze_summary)


anchor,split,fit_cutoff,rows,persistence_mean_cosine,temporal_mean_cosine,mean_cosine_lift,persistence_total_l1,temporal_total_l1,total_l1_improvement,persistence_dominant_agreement,temporal_dominant_agreement,dominant_agreement_lift
str,str,str,i64,f64,f64,f64,f64,f64,f64,f64,f64,f64
"""2025-02-13""","""other""","""2025-02-13""",139963,0.673059,0.553286,-0.119773,0.866467,1.087986,-0.221519,0.445846,0.305902,-0.139944
"""2025-03-26""","""other""","""2025-02-13""",145855,0.675406,0.494242,-0.181165,0.862884,1.168811,-0.305927,0.449392,0.253183,-0.196209
"""2025-05-07""","""other""","""2025-05-07""",149022,0.681879,0.658363,-0.023516,0.853287,0.91815,-0.064864,0.452121,0.414945,-0.037176
"""2025-06-18""","""other""","""2025-05-07""",152608,0.67884,0.632448,-0.046393,0.857396,0.963424,-0.106027,0.447447,0.387863,-0.059584
"""2025-07-30""","""primary""","""2025-07-30""",158979,0.683543,0.685948,0.002405,0.850077,0.865397,-0.015319,0.449204,0.448858,-0.000346
…,…,…,…,…,…,…,…,…,…,…,…,…
"""2025-10-22""","""primary""","""2025-10-22""",183751,0.702928,0.710503,0.007575,0.816785,0.823531,-0.006746,0.465973,0.475845,0.009872
"""2025-11-12""","""other""","""2025-10-22""",192797,0.707518,0.711938,0.00442,0.808766,0.821747,-0.01298,0.468664,0.474966,0.006302
"""2025-12-03""","""stress""","""2025-10-22""",200275,0.704422,0.711035,0.006613,0.81383,0.823429,-0.009598,0.465879,0.470144,0.004264


split,anchors,mean_cosine_lift,total_l1_improvement,dominant_agreement_lift
str,u32,f64,f64,f64
"""other""",5,-0.073285,-0.142263,-0.085322
"""primary""",4,0.007525,-0.006321,0.007953
"""stress""",2,0.006686,-0.008711,0.005382


In [32]:
BASE_EXPORT_COLS=[
    "user_id","anchor_date","aux_fit_cutoff","arch_fit_cutoff","aux_available",*[f"aux_mean_{name}" for name in CORE_STATE_NAMES],
    "future_arch_availability_score","future_arch_measurement_confidence","future_arch_quality_score","future_arch_rank_reference_all_users","k5_canonical_v31_order",
    *[f"future_arch_eb__{name}" for name in CORE_ARCH_FEATURES],*[f"future_arch_rank__{name}" for name in CORE_ARCH_FEATURES],
    *[f"future_arch5_w_{j}" for j in range(ARCH_K5)],"future_arch5_dominant","future_arch5_dominant_weight","future_arch5_entropy","future_arch5_effective_count",
    "current_arch5_weight_eligible","current_arch5_measurement_confidence",*[f"current_arch5_w_{j}" for j in range(ARCH_K5)],
    "current_arch5_dominant","current_arch5_dominant_weight","current_arch5_entropy","current_arch5_effective_count",
    *[f"future_arch5_delta_w_{j}" for j in range(ARCH_K5)],"future_vs_current_arch5_cosine","future_vs_current_arch5_l1","future_vs_current_arch5_dominant_same",
]
K6_COLS=["postny_k6_refinement_enabled","k6_basis_has_completed_postny_33d",*[f"future_arch6_w_{j}" for j in range(ARCH_K6)],
         "future_arch6_dominant","future_arch6_dominant_weight","future_arch6_entropy","future_arch6_effective_count"]

def load_final_anchor_prediction(anchor):
    p=PRED_DIR / f"future_behaviour_archetypes_v4_final_anchor_{anchor}.parquet"
    if not p.exists(): raise FileNotFoundError(p)
    return pl.read_parquet(p)

def scenario_b_frame(df):
    enabled=pl.col("postny_k6_refinement_enabled")==1; exprs=[]
    for c in K6_COLS:
        if c in {"postny_k6_refinement_enabled","k6_basis_has_completed_postny_33d"}: exprs.append(pl.col(c))
        else: exprs.append(pl.when(enabled).then(pl.col(c)).otherwise(None).alias(c))
    return df.select([*BASE_EXPORT_COLS,*exprs])

frames_k5=[]; frames_k5k6=[]
for anchor in CANONICAL_ANCHORS:
    df=load_final_anchor_prediction(anchor); frames_k5.append(df.select(BASE_EXPORT_COLS)); frames_k5k6.append(scenario_b_frame(df))
historical_k5=pl.concat(frames_k5,how="vertical_relaxed"); historical_k5k6=pl.concat(frames_k5k6,how="vertical_relaxed")
expected_rows=len(CANONICAL_ANCHORS)*EXPECTED_USERS; assert historical_k5.height==expected_rows and historical_k5k6.height==expected_rows
hist_k5_path=ARTIFACT_ROOT / "temporal_features_historical_K5.parquet"; hist_k5k6_path=ARTIFACT_ROOT / "temporal_features_historical_K5_plus_postNY_K6.parquet"
historical_k5.write_parquet(hist_k5_path,compression="zstd",statistics=True); historical_k5k6.write_parquet(hist_k5k6_path,compression="zstd",statistics=True)
final_df=load_final_anchor_prediction(FINAL_ANCHOR); final_k5=final_df.select(BASE_EXPORT_COLS); final_k5k6=scenario_b_frame(final_df)
final_k5_path=ARTIFACT_ROOT / "temporal_features_final_K5_2026-02-13.parquet"; final_k5k6_path=ARTIFACT_ROOT / "temporal_features_final_K5_plus_postNY_K6_2026-02-13.parquet"
final_k5.write_parquet(final_k5_path,compression="zstd",statistics=True); final_k5k6.write_parquet(final_k5k6_path,compression="zstd",statistics=True)
print("historical K5:",historical_k5.shape,hist_k5_path); print("historical K5+postNY K6:",historical_k5k6.shape,hist_k5k6_path)
print("final K5:",final_k5.shape,final_k5_path); print("final K5+postNY K6:",final_k5k6.shape,final_k5k6_path)


historical K5: (2750000, 80) D:\baboteek_holmotech\artifacts\prepare_features\temporal_features_historical_K5.parquet
historical K5+postNY K6: (2750000, 92) D:\baboteek_holmotech\artifacts\prepare_features\temporal_features_historical_K5_plus_postNY_K6.parquet
final K5: (250000, 80) D:\baboteek_holmotech\artifacts\prepare_features\temporal_features_final_K5_2026-02-13.parquet
final K5+postNY K6: (250000, 92) D:\baboteek_holmotech\artifacts\prepare_features\temporal_features_final_K5_plus_postNY_K6_2026-02-13.parquet


In [34]:
for forbidden in ["future_arch_weight_available_prob","future_arch_weight_eligible","future_arch_rank_reference_eligible"]:
    assert forbidden not in final_k5.columns, forbidden
for j in range(ARCH_K5):
    assert historical_k5[f"future_arch5_w_{j}"].null_count()==0 and final_k5[f"future_arch5_w_{j}"].null_count()==0
hist_ineligible=historical_k5.filter(pl.col("current_arch5_weight_eligible")==0).height
for j in range(ARCH_K5): assert historical_k5[f"current_arch5_w_{j}"].null_count()==hist_ineligible
pre_ny=historical_k5k6.filter(pl.col("anchor_date")<pl.lit(POSTNY_REFINEMENT_START))
for j in range(ARCH_K6): assert pre_ny[f"future_arch6_w_{j}"].drop_nulls().len()==0
post_ny_hist=historical_k5k6.filter(pl.col("anchor_date")>=pl.lit(POSTNY_REFINEMENT_START))
if post_ny_hist.height:
    for j in range(ARCH_K6): assert post_ny_hist[f"future_arch6_w_{j}"].null_count()==0
for j in range(ARCH_K6): assert final_k5k6[f"future_arch6_w_{j}"].null_count()==0

manifest={
    "version":"aux_future_behaviour_archetypes_v4_final","status":"FINAL_FREEZE_TEMPORAL_ARCHETYPE_POSTPROCESS",
    "source_v2_root":str(SOURCE_V2_ROOT),"output_root":str(ARTIFACT_ROOT),
    "temporal_models_retrained":False,"model_inference_rerun":False,"mean_trajectory_source":"existing V2 mean_daily_trajectories",
    "canonical_ensemble_export":"V2 mean GRU+TCN 50/50; blend sweep deferred",
    "representation":{"canonical":"K5","optional_refinement":"K6 post-NY only","k6_replaces_k5":False,"k5_canonical_from_v2_pit":K5_CANONICAL_FROM_V2_PIT.tolist()},
    "future_trait_semantics":{"hard_eligibility_gate":False,"trait_columns_null_when_predicted_low_evidence":False,
        "rank_reference":"all predicted users; frozen V3 OOT geometry","availability_feature":"raw model-derived score; NOT treated as calibrated probability",
        "quality_score":"raw availability score * EB/activity measurement confidence"},
    "current_persistence_semantics":{"window":"observed [anchor-29d, anchor]","current_k5_actual_evidence_gate":"active_days >= 3",
        "current_k5_null_when_insufficient_actual_evidence":True,"export_current_k5":True,"export_future_minus_current_k5":True,"export_change_cosine_l1_dominant_same":True},
    "corrected_windows":{"pre_ny":"2025-11-13..2025-12-14 (32d)","ny_regime_excluded":"2025-12-15..2026-01-11","post_ny":"2026-01-12..2026-02-13 (33d)"},
    "exports":{"historical_k5":str(hist_k5_path),"historical_k5_plus_postny_k6":str(hist_k5k6_path),"final_k5":str(final_k5_path),"final_k5_plus_postny_k6":str(final_k5k6_path)},
    "no_gmv_model":True,"no_catboost":True,
}
manifest_path=REPORT_DIR / "v4_final_manifest.json"; manifest_path.write_text(json.dumps(manifest,ensure_ascii=False,indent=2),encoding="utf-8")
use_these=ARTIFACT_ROOT / "USE_THESE_FINAL_FILES.txt"; use_these.write_text("\n".join([
    "GMV future behaviour archetypes — V4 FINAL","","Use these files downstream:",str(hist_k5_path),str(hist_k5k6_path),str(final_k5_path),str(final_k5k6_path),
    "","Canonical: K5","Optional experiment: K5 + post-NY K6","Do NOT use V3 parquet as final production inputs."]),encoding="utf-8")
report_zip=ARTIFACT_ROOT / "v4_final_reports.zip"
with zipfile.ZipFile(report_zip,"w",compression=zipfile.ZIP_DEFLATED) as zf:
    for p in sorted(REPORT_DIR.glob("*")):
        if p.is_file(): zf.write(p,arcname=f"reports/{p.name}")
    zf.write(use_these,arcname=use_these.name)
print("\nV4 FINAL SANITY: OK"); print("manifest:",manifest_path); print("reports:",report_zip); print("use-these:",use_these)



V4 FINAL SANITY: OK
manifest: D:\baboteek_holmotech\artifacts\prepare_features\reports\v4_final_manifest.json
reports: D:\baboteek_holmotech\artifacts\prepare_features\v4_final_reports.zip
use-these: D:\baboteek_holmotech\artifacts\prepare_features\USE_THESE_FINAL_FILES.txt
